# GNN Powerflow V2.6 — Training

Loads network datasets, runs hyperparameter sweeps, saves results.

**Input**: dataset JSONs from `GNN_Powerflow_V2.6_DataGen.ipynb`

**Output**: sweep result JSONs in `TRAINING_RESULTS_DIR`

In [ ]:
# %%
import copy
import logging
import os
import pickle
import random
import time
from typing import Dict, List, Optional, Tuple
import json
from datetime import datetime
from dataclasses import dataclass, field, asdict

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pypsa
import pypowsybl as pp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch_geometric.data import Batch, Data, Dataset
from torch_geometric.nn import GATv2Conv, GCNConv, GraphConv
from tqdm import tqdm

import urllib

import statsmodels.api as sm
from statsmodels.formula.api import ols

# Suppress verbose output from PyPSA and dependencies during data generation
for _log in ("pypsa", "pypsa.pf", "pypsa.components", "numexpr", "linopy"):
    logging.getLogger(_log).setLevel(logging.WARNING)
logging.getLogger("numexpr").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

import re
from scipy import ndimage

In [ ]:
# ── Configurable paths ───────────────────────────────────────────────────────
# Set DATA_ROOT to wherever your data folders live.
# Use "." to keep everything relative to the repo (default).
# Example: DATA_ROOT = r"C:\Users\you\OneDrive\Data_PF_GNN"
import os as _os
#DATA_ROOT             = r"C:\Users\STSI\OneDrive - USN\Data_PF_GNN"                                             
DATA_ROOT             = r"C:\Users\ssimo\OneDrive - USN\Data_PF_GNN"                                             
GRID_MODEL_FILES      = _os.path.join(DATA_ROOT, "grid_model_files")    # static grid CSV/RAW files
TRAINING_NETWORKS_DIR = _os.path.join(DATA_ROOT, "training_networks_saved")  # dataset JSONs
TRAINING_RESULTS_DIR  = _os.path.join(DATA_ROOT, "training_results_saved")   # sweep result JSONs/CSVs


In [ ]:
#test and execution toggles

run_regression_test=False # to run a regression test on the GNN training loop (with very limited epochs and data)


In [ ]:
import itertools

# ── Style pools ──────────────────────────────────────────────────────────────
_LINE_STYLES  = ["-", "--", "-.", ":"]
_MARKERS      = ["", "o", "s", "^", "D", "v", "P", "X"]
_COLOR_CYCLE  = plt.rcParams["axes.prop_cycle"].by_key()["color"]

## Data Generation Functions
_Paste from V2.5.3 cells 20, 22, 23, 25, 26_

In [ ]:
#==================save and load genrated datasets========================

# not needed
def save_dataset_list(
    dataset_list: list,
    index_path: str,
    tag: str = "dataset",
    subfolder: str | None = None,
) -> str:
    """
    Save a list of PyPSA networks using NetCDF (avoids weakref pickle errors).

    Each network is saved as an individual .nc file inside a dedicated subfolder;
    an index JSON records the ordered list of paths so they can be reloaded in
    the same order.

    Parameters
    ----------
    dataset_list : list[pypsa.Network]
    index_path   : str        – path for the JSON index file
    tag          : str        – prefix for individual network filenames
    subfolder    : str | None – name of the subdirectory inside the index
                                directory that holds the .nc files.
                                Defaults to ``tag`` when not supplied.
                                Example: ``subfolder="dataset_118"``

    Returns
    -------
    str – path to the JSON index file
    """
    import json

    save_dir = os.path.dirname(index_path) or "."
    os.makedirs(save_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # .nc files go in their own subfolder to avoid cluttering save_dir
    nc_subdir = subfolder if subfolder is not None else tag
    nc_dir = os.path.join(save_dir, nc_subdir)
    os.makedirs(nc_dir, exist_ok=True)

    nc_paths: list[str] = []
    for i, net in enumerate(dataset_list):
        nc_name = f"{tag}_{timestamp}_{i:04d}.nc"
        nc_path = os.path.join(nc_dir, nc_name)
        net.export_to_netcdf(nc_path)
        nc_paths.append(os.path.join(nc_subdir, nc_name))  # relative to save_dir

    with open(index_path, "w") as f:
        json.dump({"tag": tag, "timestamp": timestamp, "files": nc_paths}, f, indent=2)

    print(f"Saved {len(nc_paths)} networks → {nc_dir}/ (index: {index_path})")
    return index_path

# needed
def load_dataset_list(index_path: str) -> list:
    """
    Reload a list of PyPSA networks saved by save_dataset_list.
    """
    import json

    with open(index_path, "r") as f:
        meta = json.load(f)

    save_dir = os.path.dirname(index_path) or "."
    networks: list[pypsa.Network] = []
    for nc_name in meta["files"]:
        nc_path = os.path.join(save_dir, nc_name)
        networks.append(pypsa.Network(nc_path))

    print(f"Loaded {len(networks)} networks from {index_path}")
    return networks



## Power System Helpers

In [ ]:
# =============================================================================
# SECTION 2: PTDF COMPUTATION
# =============================================================================

def compute_ptdf_matrix(network):
    """
    Compute the Power Transfer Distribution Factor (PTDF) matrix for a PyPSA network.
    Returns a DataFrame indexed by branches, columns by buses.
    """
    buses = list(network.buses.index)
    lines = list(network.lines.index)
    n_buses = len(buses)
    n_lines = len(lines)

    bus_to_idx = {b: i for i, b in enumerate(buses)}

    # Build susceptance matrix B and branch-bus incidence A
    B = np.zeros((n_buses, n_buses))
    A = np.zeros((n_lines, n_buses))

    for i, line in enumerate(lines):
        b_val = 1.0 / network.lines.loc[line, "x"] if network.lines.loc[line, "x"] != 0 else 0.0
        from_idx = bus_to_idx[network.lines.loc[line, "bus0"]]
        to_idx = bus_to_idx[network.lines.loc[line, "bus1"]]
        B[from_idx, from_idx] += b_val
        B[to_idx, to_idx] += b_val
        B[from_idx, to_idx] -= b_val
        B[to_idx, from_idx] -= b_val
        A[i, from_idx] = b_val
        A[i, to_idx] = -b_val

    # Identify slack bus (first generator with control='Slack')
    slack_bus = None
    for gen in network.generators.index:
        if network.generators.loc[gen, "control"] == "Slack":
            slack_bus = network.generators.loc[gen, "bus"]
            break
    if slack_bus is None:
        slack_bus = buses[0]
    slack_idx = bus_to_idx[slack_bus]

    # Reduce B matrix (remove slack row/col)
    non_slack = [i for i in range(n_buses) if i != slack_idx]
    B_red = B[np.ix_(non_slack, non_slack)]
    A_red = A[:, non_slack]

    try:
        B_red_inv = np.linalg.inv(B_red)
    except np.linalg.LinAlgError:
        B_red_inv = np.linalg.pinv(B_red)

    PTDF_red = A_red @ B_red_inv  # (n_lines, n_buses-1)

    # Re-insert slack column as zeros
    PTDF = np.zeros((n_lines, n_buses))
    non_slack_col = 0
    for j in range(n_buses):
        if j != slack_idx:
            PTDF[:, j] = PTDF_red[:, non_slack_col]
            non_slack_col += 1

    return PTDF



# =============================================================================
# SECTION 3: ADMITTANCE MATRIX
# =============================================================================

def compute_admittance_matrix(network, device="cpu", return_format="torch"):
    """
    Compute bus admittance matrix Y for a PyPSA network.

    Improvements over original:
      - Handles arbitrary bus naming/ordering
      - Includes line shunt b/2 at both ends
      - Includes transformer series impedance and tap_ratio
      - GPU-capable PyTorch tensors
      - Optional complex or split format

    Args:
        network: PyPSA network object
        device:  Device for PyTorch tensors ('cpu' or 'cuda')
        return_format: 'torch' -> (Y_real, Y_imag) as torch tensors
                       'numpy' -> complex numpy array
                       'complex_torch' -> complex torch tensor
    """
    num_buses = len(network.buses)
    buses = list(network.buses.index)
    bus_to_idx = {b: i for i, b in enumerate(buses)}

    Y = np.zeros((num_buses, num_buses), dtype=complex)

    # --------------------
    # Lines: π-model
    # --------------------
    for line in network.lines.index:
        row = network.lines.loc[line]
        from_bus = row["bus0"]
        to_bus   = row["bus1"]
        r = row["r"]
        x = row["x"]
        b = row["b"] if "b" in network.lines.columns else 0.0

        z = complex(r, x)
        if abs(z) < 1e-12:
            # avoid division by zero; skip or handle specially
            continue
        y_series = 1.0 / z
        y_shunt  = complex(0.0, b / 2.0)  # b is total line charging susceptance

        fi = bus_to_idx[from_bus]
        ti = bus_to_idx[to_bus]

        # series
        Y[fi, fi] += y_series
        Y[ti, ti] += y_series
        Y[fi, ti] -= y_series
        Y[ti, fi] -= y_series

        # shunts at both ends
        Y[fi, fi] += y_shunt
        Y[ti, ti] += y_shunt

    # --------------------
    # Transformers: series + tap ratio (no shunt for now)
    # --------------------
    # Transformers: use x_pu_eff/r_pu_eff (system base) instead of raw x/r
    for trafo in network.transformers.index:
        row = network.transformers.loc[trafo]
        from_bus = row["bus0"]
        to_bus   = row["bus1"]

        # STSI260320: raw x/r are on transformer MVA base; x_pu_eff/r_pu_eff
        # are already converted to system base — use these to match PyPSA's Ybus
        if "x_pu_eff" in row.index and pd.notna(row["x_pu_eff"]) and float(row["x_pu_eff"]) != 0.0:
            x = float(row["x_pu_eff"])
            r = float(row["r_pu_eff"]) if "r_pu_eff" in row.index and pd.notna(row["r_pu_eff"]) else 0.0
        else:
            x = float(row["x"])
            r = float(row.get("r", 0.0))

        tap = float(row["tap_ratio"]) if "tap_ratio" in row.index else 1.0

        z = complex(r, x)
        if abs(z) < 1e-12:
            continue
        y_series = 1.0 / z

        fi = bus_to_idx[from_bus]
        ti = bus_to_idx[to_bus]
        t  = complex(tap, 0.0)

        Y[fi, fi] += y_series / (t * t.conjugate())
        Y[ti, ti] += y_series
        Y[fi, ti] -= y_series / t.conjugate()
        Y[ti, fi] -= y_series / t


    # ---------------
    # Return formats
    # ---------------
    if return_format == "numpy":
        return Y
    elif return_format == "complex_torch":
        return torch.tensor(Y, dtype=torch.complex64, device=device)
    else:  # 'torch' -> split real/imag
        Y_real = torch.tensor(Y.real, dtype=torch.float32, device=device)
        Y_imag = torch.tensor(Y.imag, dtype=torch.float32, device=device)
        return Y_real, Y_imag

def get_pypsa_Y_numpy(net: pypsa.Network) -> np.ndarray | None:
    """
    Get PyPSA's Ybus via subnetwork API.

    STSI260320: Works on a deepcopy to avoid mutating training networks.
    determine_network_topology() modifies net.buses['sub_network'] in-place,
    which can corrupt dataset construction if called on live training networks.
    """
    if hasattr(net, "admittance_matrix"):
        Y = net.admittance_matrix()
        return np.asarray(Y.todense())

    if not hasattr(net, "determine_network_topology"):
        return None

    try:
        # STSI260320: deepcopy to prevent mutation of training network objects
        net_copy = copy.deepcopy(net)
        net_copy.determine_network_topology()
    except Exception as e:
        logger.debug(f"[YCHECK] determine_network_topology failed: {e}")
        return None

    if not hasattr(net_copy, "sub_networks") or len(net_copy.sub_networks) == 0:
        return None

    sn_label = list(net_copy.sub_networks.index)[0]
    sn = net_copy.sub_networks.at[sn_label, "obj"]

    if not hasattr(sn, "calculate_Y"):
        return None

    try:
        sn.calculate_Y()
        Y_sparse = sn.Y
        Y_dense  = np.asarray(Y_sparse.todense())

        sn_bus_order  = list(sn.buses_i())
        net_bus_order = list(range(len(net_copy.buses)))

        if sorted(sn_bus_order) == net_bus_order:
            return Y_dense[np.ix_(sn_bus_order, sn_bus_order)]
        else:
            return Y_dense

    except Exception as e:
        logger.debug(f"[YCHECK] calculate_Y() failed: {e}")
        return None

def precompute_Y_matrices(
    networks: list,
    device: str | torch.device = "cpu",
    y_tolerance: float = 1e-6,
    y_matrix_source: str = "auto",   # STSI260320: "manual", "pypsa", "auto"
):
    """
    Pre-compute Y matrices for a list of networks.

    y_matrix_source:
      "manual" — always use compute_admittance_matrix (your implementation).
      "pypsa"  — always use PyPSA's subnetwork Y; raise if not available.
      "auto"   — use PyPSA Y if available and consistent within y_tolerance,
                 otherwise fall back to manual Y and log a warning.

    STSI260320: Added y_matrix_source argument so this can be controlled as
    a hyperparameter in run_hparam_sweep, allowing ablation between manual
    and PyPSA-derived Y matrices in the physics loss.
    """
    assert y_matrix_source in ("manual", "pypsa", "auto"), \
        f"y_matrix_source must be 'manual', 'pypsa', or 'auto', got {y_matrix_source!r}"

    if isinstance(device, str):
        device = torch.device(device)

    Y_list  = []
    n_manual = 0
    n_pypsa  = 0
    n_diff   = 0

    for idx, net in enumerate(networks):
        Y_manual = compute_admittance_matrix(net, device=device, return_format="numpy")
        Y_pypsa  = get_pypsa_Y_numpy(net)  # None if unavailable

        if y_matrix_source == "manual":
            Y_use = Y_manual
            n_manual += 1

        elif y_matrix_source == "pypsa":
            if Y_pypsa is None:
                raise RuntimeError(
                    f"y_matrix_source='pypsa' but PyPSA Y not available for network {idx}."
                )
            Y_use = Y_pypsa
            n_pypsa += 1

        else:  # "auto"
            if Y_pypsa is None:
                logger.debug(f"[YCHECK] Network {idx}: PyPSA Y unavailable — using manual.")
                Y_use = Y_manual
                n_manual += 1
            elif Y_manual.shape != Y_pypsa.shape:
                logger.warning(
                    f"[YCHECK] Network {idx}: shape mismatch "
                    f"{Y_manual.shape} vs {Y_pypsa.shape} — using PyPSA Y."
                )
                Y_use = Y_pypsa
                n_pypsa += 1
                n_diff  += 1
            else:
                max_diff = np.max(np.abs(Y_manual - Y_pypsa))
                if max_diff > y_tolerance:
                    logger.warning(
                        f"[YCHECK] Network {idx}: max|Y_manual-Y_pypsa|={max_diff:.3e} "
                        f"> tol={y_tolerance:.1e} — using PyPSA Y."
                    )
                    Y_use = Y_pypsa
                    n_pypsa += 1
                    n_diff  += 1
                else:
                    logger.debug(
                        f"[YCHECK] Network {idx}: manual Y matches PyPSA "
                        f"(max diff {max_diff:.3e})."
                    )
                    Y_use = Y_manual
                    n_manual += 1

        Y_real = torch.from_numpy(Y_use.real).to(device=device, dtype=torch.float32)
        Y_imag = torch.from_numpy(Y_use.imag).to(device=device, dtype=torch.float32)
        Y_list.append((Y_real, Y_imag))

    # STSI260320: single summary log instead of per-network messages
    if n_diff > 0:
        logger.warning(
            f"[YCHECK] Summary ({y_matrix_source}): {n_pypsa} PyPSA Y, "
            f"{n_manual} manual Y, {n_diff} had mismatches > {y_tolerance:.1e}."
        )
    else:
        logger.info(
            f"[YCHECK] Summary ({y_matrix_source}): {n_pypsa} PyPSA Y, "
            f"{n_manual} manual Y, 0 mismatches."
        )

    return Y_list


## GNN Architecture

In [ ]:
# =============================================================================
# SECTION 4: DATASET
# =============================================================================

import torch
from torch_geometric.data import Data, Dataset

class PowerFlowDataset(Dataset):
    """
    One data object per (network, snapshot) pair.

    The graph topology reflects the physical network topology:
    - buses → GNN nodes
    - lines + transformers → GNN edges (bidirectional)

    Each graph carries:
      x         : [n_buses, 7 or 8] node features (bus type flags, known P/Q/V, optional p_net_balance)
      y         : [n_buses, 4]       targets (vmag, vang, p, q)
      edge_index: [2, n_edges]       bidirectional
      edge_attr : [n_edges, 6]       edge features (r, x, b/2, tap, g_series, b_series)
      masks     : slack_mask, pv_mask, pq_mask
      y_line_p  : [n_fwd_lines]      forward-direction real power per supervised edge (PTDF supervision)
      ptdf_edge_row_idx : [n_edges]  PTDF row index for each edge (-1 for non-supervised/reverse)  # STSI 26.04.07
      network_idx: int
    """

    def __init__(
        self,
        networks: list,
        use_edge_features: bool = True,
        use_pnet_balance: bool = True,   # Step 1: append p_net_balance to node features
        transform=None,
        pre_transform=None,
        ptdf_branch_mode: str = "lines",  # "lines" or "all" — controls PTDF supervision scope
    ):
        super().__init__(root=None, transform=transform, pre_transform=pre_transform)
        self.networks          = networks
        self.use_edge_features = use_edge_features
        self.use_pnet_balance  = use_pnet_balance
        self.ptdf_branch_mode  = ptdf_branch_mode #STSI260402: pass ptdf_branch_mode to _create_graph_data for edge feature control

        # Build flat index: list of (network_idx, timestep_idx)
        self._index = []
        for net_idx, net in enumerate(networks):
            for t_idx in range(len(net.snapshots)):
                self._index.append((net_idx, t_idx))

    def len(self):
        return len(self._index)

    def get(self, idx):
        net_idx, t_idx = self._index[idx]
        network = self.networks[net_idx]
        data = self._create_graph_data(network, t_idx, net_idx)
        return data

    def _create_graph_data(self, network: pypsa.Network, t_idx: int, net_idx: int) -> Data:
        buses     = list(network.buses.index)
        n_buses   = len(buses)
        bus_to_i  = {b: i for i, b in enumerate(buses)}
        snapshot  = network.snapshots[t_idx]

        # ── Bus type flags ────────────────────────────────────────────────────
        is_slack = torch.zeros(n_buses, dtype=torch.float)
        is_pv    = torch.zeros(n_buses, dtype=torch.float)
        is_pq    = torch.zeros(n_buses, dtype=torch.float)

        slack_mask = torch.zeros(n_buses, dtype=torch.bool)
        pv_mask    = torch.zeros(n_buses, dtype=torch.bool)
        pq_mask    = torch.zeros(n_buses, dtype=torch.bool)

        for gen_name, gen in network.generators.iterrows():
            bus_name = gen["bus"]
            if bus_name not in bus_to_i:
                continue
            i = bus_to_i[bus_name]
            ctrl = gen["control"]
            if ctrl == "Slack":
                is_slack[i] = 1.0
                slack_mask[i] = True
            elif ctrl == "PV":
                is_pv[i] = 1.0
                pv_mask[i] = True

        # Buses with no generator → PQ
        for i in range(n_buses):
            if not slack_mask[i] and not pv_mask[i]:
                is_pq[i]   = 1.0
                pq_mask[i] = True

        # ── Bus injections (P, Q) from PyPSA post-solve ───────────────────────
        # STSI260404: BUGFIX — use .loc[snapshot, buses] to force column order
        # to match network.buses.index.  Without this, buses_t columns may be
        # in lexicographic order (e.g. "Bus 10" before "Bus 2"), mis-aligning
        # P/Q/Vang with bus-type masks and the Y-matrix built from buses.index.
        p_bus = torch.tensor(
            network.buses_t.p.loc[snapshot, buses].values, dtype=torch.float
        )
        q_bus = torch.tensor(
            network.buses_t.q.loc[snapshot, buses].values, dtype=torch.float
        )

        # ── Voltages ─────────────────────────────────────────────────────────────
        v_mag = torch.tensor(
            network.buses_t.v_mag_pu.loc[snapshot, buses].values, dtype=torch.float
        )
        v_ang = torch.tensor(
            network.buses_t.v_ang.loc[snapshot, buses].values, dtype=torch.float
        )

        # ── Input features: GNN receives known values, zeros where unknown ────
        # Known before solve: bus type flags, P_inj for PQ+PV, Q_inj for PQ,
        #                     Vmag for PV+Slack, Vang for Slack (=0)
        #
        # x layout: [is_slack(0), is_PV(1), is_PQ(2), P(3), Q(4), Vmag(5), Vang(6)]
        #
        # NOTE: Slack P input x[:,3]=0 is a known issue — the model sees no
        # varying signal for slack P. This is partially addressed by p_net_balance.
        x_p    = p_bus.clone()
        x_q    = q_bus.clone()
        x_vmag = v_mag.clone()
        x_vang = v_ang.clone()

        # Mask unknowns to zero in input (prevents trivial leakage)
        x_p[slack_mask]    = 0.0   # Slack P unknown (to be predicted)
        x_q[slack_mask]    = 0.0   # Slack Q unknown
        x_q[pv_mask]       = 0.0   # PV Q unknown
        x_vmag[pq_mask]    = 0.0   # PQ Vmag unknown
        x_vang[pq_mask]    = 0.0   # PQ Vang unknown
        x_vang[pv_mask]    = 0.0   # PV Vang unknown

        x = torch.stack([is_slack, is_pv, is_pq, x_p, x_q, x_vmag, x_vang], dim=1)

        # ── Step 1: p_net_balance — graph-level scalar for slack signal ───────
        # Net active power balance = sum of all bus injections at this snapshot.
        # This gives the slack bus a varying signal correlated with its true P.
        p_net_balance = float(p_bus.sum().item())
        p_net_bal_tensor = torch.tensor([p_net_balance], dtype=torch.float)

        if self.use_pnet_balance:
            # Broadcast p_net_balance to all nodes as an 8th feature
            p_net_col = p_net_bal_tensor.expand(n_buses, 1)
            x = torch.cat([x, p_net_col], dim=1)

        # ── Target: full post-solve state for all buses ──────────────────────────
        y = torch.stack([v_mag, v_ang, p_bus, q_bus], dim=1)

        # ── Edge construction (lines + transformers) ──────────────────────────────
        edge_index_list = []
        edge_attr_list  = []
        forward_edge_mask = []   # True for first direction (not reverse duplicate)
        fwd_line_idx_list  = []   # one entry per forward supervised edge
        ptdf_edge_row_idx_list = []  # STSI 26.04.07: PTDF row index per edge for learned_edge mode

        branches = []
        #STSI260402: control which branches are included in PTDF supervision via self.ptdf_branch_mode
        # lines always included
        for line_name, line in network.lines.iterrows():
            if line["bus0"] in bus_to_i and line["bus1"] in bus_to_i:
                branches.append(
                    (
                        line.name,  # name (not used in PTDF but useful for debugging)
                        line["bus0"], line["bus1"],
                        line["r"], line["x"],
                        line.get("b", 0.0), 1.0, "line"
                    )
                )

        # STSI260404: BUGFIX — transformers must ALWAYS be added as graph edges
        # for GNN message passing, regardless of ptdf_branch_mode.
        # ptdf_branch_mode only controls which branches are supervised by PTDF
        # loss (via forward_edge_mask / fwd_line_idx_list), NOT graph topology.
        # Previously gated behind `if self.ptdf_branch_mode == "all":` which
        # silently severed the graph when mode was "lines" (the default).
        for tr_name, tr in network.transformers.iterrows():
            if tr["bus0"] in bus_to_i and tr["bus1"] in bus_to_i:
                tap = tr.get("tap_ratio", 1.0)
                tap = tap if not pd.isna(tap) else 1.0
                branches.append(
                    (
                        tr.name,  # name (not used in PTDF but useful for debugging)
                        tr["bus0"], tr["bus1"],
                        tr["r"], tr["x"],
                        tr.get("b", 0.0), float(tap), "trafo"
                    )
                )


        for branch_idx, (name, b0, b1, r, x_val, b_val, tap, btype) in enumerate(branches):
            i0, i1 = bus_to_i[b0], bus_to_i[b1]
            z      = complex(r, x_val)
            y_ser  = (1.0 / z) if abs(z) > 1e-12 else 0.0
            g_ser, b_ser = y_ser.real, y_ser.imag

            if self.use_edge_features:
                attr = [r, x_val, b_val / 2.0, tap, g_ser, b_ser]
            else:
                attr = [1.0]

            # Forward edge
            edge_index_list.append([i0, i1])
            edge_attr_list.append(attr)

            # PTDF supervision: only lines (or all branches) get True in forward_edge_mask
            is_supervised = (btype == "line") if self.ptdf_branch_mode == "lines" else True
            forward_edge_mask.append(is_supervised)
            if is_supervised:
                fwd_line_idx_list.append(branch_idx)
                ptdf_edge_row_idx_list.append(branch_idx)  # STSI 26.04.07: supervised forward edge
            else:
                ptdf_edge_row_idx_list.append(-1)  # STSI 26.04.07: unsupervised forward edge

            # Reverse edge
            edge_index_list.append([i1, i0])
            edge_attr_list.append(attr)
            forward_edge_mask.append(False)
            ptdf_edge_row_idx_list.append(-1)  # STSI 26.04.07: reverse edge

        if edge_index_list:
            edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
            edge_attr  = torch.tensor(edge_attr_list,  dtype=torch.float)
            fwd_mask   = torch.tensor(forward_edge_mask, dtype=torch.bool)
            line_idx   = torch.tensor(fwd_line_idx_list, dtype=torch.long)
            ptdf_edge_row_idx = torch.tensor(ptdf_edge_row_idx_list, dtype=torch.long)  # STSI 26.04.07
        else:
            edge_index = torch.zeros((2, 0), dtype=torch.long)
            edge_attr  = torch.zeros((0, len(attr) if branches else 1), dtype=torch.float)
            fwd_mask   = torch.zeros(0, dtype=torch.bool)
            line_idx   = torch.zeros(0, dtype=torch.long)
            ptdf_edge_row_idx = torch.zeros(0, dtype=torch.long)  # STSI 26.04.07


        try:
            ptdf_matrix = compute_ptdf_matrix(network)
            y_ptdf = torch.tensor(ptdf_matrix, dtype=torch.float)
        except Exception:
            n_lines = len(network.lines)
            y_ptdf = torch.zeros((n_lines, n_buses), dtype=torch.float)

        # STSI260402 Assertions — verify PTDF supervision is lines-only and consistent with y_ptdf
        n_lines = len(network.lines)
        assert int(fwd_mask.sum()) == n_lines, \
            f"forward_edge_mask true count {int(fwd_mask.sum())} != n_lines {n_lines}"
        assert y_ptdf.shape[0] == n_lines, \
            f"y_ptdf rows {y_ptdf.shape[0]} != n_lines {n_lines}"

        # ── Line active power flows ───────────────────────────────────────────
        if not network.lines_t.p0.empty:
            y_line_p = torch.tensor(
                network.lines_t.p0.loc[snapshot].values, dtype=torch.float
            )
        else:
            y_line_p = torch.zeros(len(network.lines), dtype=torch.float)

        # ── Assemble Data object ──────────────────────────────────────────────
        data = Data(
            x          = x,
            y          = y,
            edge_index = edge_index,
            edge_attr  = edge_attr,
        )

        # Bus-type masks — node-level bool tensors (critical for collate + loss)
        data.slack_mask = slack_mask   # [n_nodes] bool
        data.pv_mask    = pv_mask      # [n_nodes] bool
        data.pq_mask    = pq_mask      # [n_nodes] bool

        # Variable-size targets (handled specially in collate_with_ptdf)
        data.y_ptdf      = y_ptdf
        data.y_line_p    = y_line_p
        data.forward_edge_mask = fwd_mask
        data.ptdf_line_index = line_idx
        data.ptdf_edge_row_idx = ptdf_edge_row_idx  # STSI 26.04.07: per-edge PTDF row index for learned_edge mode

        # Graph-level metadata
        data.network_idx    = torch.tensor([net_idx], dtype=torch.long)
        data.p_net_balance  = p_net_bal_tensor   # Step 1: slack signal

        return data

def collate_with_ptdf(batch):
    """
    Custom collate function for variable-size PTDF matrices and per-graph attributes.
    
    FIX 2026-03-25: Explicitly propagate slack_mask, pv_mask, pq_mask so that
    _masked_mse_loss and physics_informed_loss_batch receive valid mask tensors.
    Without this, all masked losses silently return 0.
    
    Also propagates p_net_balance (scalar per graph, Step 1 slack signal fix).
    """
    # ── Variable-size lists: remove before PyG batching ──────────────────────
    y_ptdf_list   = [data.y_ptdf   for data in batch]
    y_line_p_list = [data.y_line_p for data in batch]
    ptdf_line_index_list = [data.ptdf_line_index for data in batch]


    for data in batch:
        del data.y_ptdf
        del data.y_line_p #STSI260402: added deletion of y_line_p to prevent PyG from trying to batch it and causing errors due to variable sizes
        del data.ptdf_line_index
    # ── PyG auto-batches all node-level tensors (x, y, edge_index, etc.) ─────
    # slack_mask / pv_mask / pq_mask are node-level bool tensors stored on each
    # Data object by PowerFlowDataset — Batch.from_data_list will concat them
    # along the node dimension automatically IF they are present.
    # ptdf_edge_row_idx is edge-level [n_edges] long — also auto-concatenated (STSI 26.04.07)
    batched = Batch.from_data_list(batch)

    # ── Restore variable-size list attributes ────────────────────────────────
    batched.y_ptdf_list   = y_ptdf_list
    batched.y_line_p_list = y_line_p_list
    batched.ptdf_line_index_list = ptdf_line_index_list

    # Restore on originals for re-use in dataset
    for data, y_ptdf, y_line_p, ptdf_idx in zip(batch, y_ptdf_list, y_line_p_list, ptdf_line_index_list):
        data.y_ptdf   = y_ptdf
        data.y_line_p = y_line_p
        data.ptdf_line_index = ptdf_idx

    # ── Defensive check: abort loudly if masks are missing ───────────────────
    for mask_attr in ("slack_mask", "pv_mask", "pq_mask"):
        if not hasattr(batched, mask_attr):
            raise RuntimeError(
                f"collate_with_ptdf: batched object missing '{mask_attr}'. "
                f"Ensure PowerFlowDataset stores data.{mask_attr} as a node-level "
                f"bool tensor (shape [n_nodes]) on every Data object."
            )

    return batched

# training data validation
def validate_training_data(networks: list, name: str = "dataset", n_sample: int = 2) -> bool:
    """
    Validate that a list of PyPSA networks is compatible with PowerFlowDataset.
    Checks all fields accessed by _create_graph_data and compute_ptdf_matrix.

    Parameters
    ----------
    networks : list of pypsa.Network (post-PF)
    name     : label for print output
    n_sample : how many networks to inspect in detail

    Returns
    -------
    bool : True if all checks pass, False if any critical issue found
    """
    import numpy as np
    import torch

    issues   = []   # critical — will break the loader
    warnings = []   # non-critical — may degrade training

    print(f"\n{'='*60}")
    print(f"Validating '{name}': {len(networks)} networks")
    print(f"{'='*60}")

    if len(networks) == 0:
        print("CRITICAL: Empty network list")
        return False

    for net_idx, net in enumerate(networks[:n_sample]):
        tag = f"[net {net_idx}]"
        print(f"\n--- Network {net_idx} ---")

        # ── 1. Topology ──────────────────────────────────────────
        n_buses  = len(net.buses)
        n_lines  = len(net.lines)
        n_trafos = len(net.transformers)
        n_gens   = len(net.generators)
        n_loads  = len(net.loads)
        n_snaps  = len(net.snapshots)
        print(f"  Buses={n_buses}, Lines={n_lines}, Trafos={n_trafos}, "
              f"Gens={n_gens}, Loads={n_loads}, Snapshots={n_snaps}")

        if n_buses == 0:
            issues.append(f"{tag} No buses")
        if n_lines + n_trafos == 0:
            issues.append(f"{tag} No lines or transformers — edge_index will be empty")
        if n_snaps == 0:
            issues.append(f"{tag} No snapshots — dataset will be empty")
        if n_gens == 0:
            warnings.append(f"{tag} No generators — bus type will default to PQ everywhere")

        # ── 2. Required buses_t tables (read in _create_graph_data) ──
        print(f"\n  buses_t tables:")
        for tbl, attr in [("buses_t.p",       "p"),
                           ("buses_t.q",       "q"),
                           ("buses_t.v_mag_pu","v_mag_pu"),
                           ("buses_t.v_ang",   "v_ang")]:
            df = getattr(net.buses_t, attr)
            ok = not df.empty and df.shape == (n_snaps, n_buses)
            finite = np.isfinite(df.values).all() if ok else False
            status = "✅" if (ok and finite) else "❌"
            if not ok:
                issues.append(f"{tag} {tbl} missing or wrong shape: {df.shape} "
                              f"(expect {(n_snaps, n_buses)})")
            elif not finite:
                issues.append(f"{tag} {tbl} contains NaN/Inf")
            print(f"    {status} {tbl}: shape={df.shape}, finite={finite}")

        # ── 3. Voltage range sanity ───────────────────────────────
        if not net.buses_t.v_mag_pu.empty:
            v = net.buses_t.v_mag_pu.values.flatten()
            v_ok = np.all((v > 0.5) & (v < 1.5))
            status = "✅" if v_ok else "⚠️"
            print(f"    {status} v_mag_pu range: [{v.min():.4f}, {v.max():.4f}]")
            if not v_ok:
                warnings.append(f"{tag} v_mag_pu out of [0.5, 1.5]: "
                                f"min={v.min():.4f} max={v.max():.4f}")

        # ── 4. lines_t.p0 (used for y_line_p) ────────────────────
        print(f"\n  lines_t tables:")
        p0 = net.lines_t.p0
        ok_p0 = not p0.empty and p0.shape == (n_snaps, n_lines)
        finite_p0 = np.isfinite(p0.values).all() if ok_p0 else False
        status = "✅" if (ok_p0 and finite_p0) else "❌"
        if not ok_p0:
            issues.append(f"{tag} lines_t.p0 missing or wrong shape: {p0.shape} "
                         f"(expect {(n_snaps, n_lines)})")
        elif not finite_p0:
            issues.append(f"{tag} lines_t.p0 contains NaN/Inf")
        print(f"    {status} lines_t.p0: shape={p0.shape}, finite={finite_p0}")

        # ── 5. generators_t.p and q (used for node features) ─────
        print(f"\n  generators_t tables:")
        for attr in ["p", "q"]:
            df = getattr(net.generators_t, attr)
            ok = not df.empty
            finite = np.isfinite(df.values).all() if ok else False
            status = "✅" if (ok and finite) else "⚠️"
            if not ok:
                warnings.append(f"{tag} generators_t.{attr} empty")
            print(f"    {status} generators_t.{attr}: shape={df.shape}, finite={finite}")

        # ── 6. Generator control column ───────────────────────────
        print(f"\n  Generator control types:")
        if "control" not in net.generators.columns:
            issues.append(f"{tag} net.generators missing 'control' column")
        else:
            ctrl_counts = net.generators["control"].value_counts().to_dict()
            has_slack = "Slack" in ctrl_counts
            status = "✅" if has_slack else "❌"
            print(f"    {status} control types: {ctrl_counts}")
            if not has_slack:
                issues.append(f"{tag} No Slack generator — bus type encoding will be wrong")

        # ── 7. Line parameters (edge_attr: r, x, b, s_nom) ───────
        print(f"\n  Line parameters:")
        for col in ["r", "x", "b", "s_nom"]:
            present = col in net.lines.columns
            if present:
                vals = net.lines[col].values.astype(float)
                finite = np.isfinite(vals).all()
                status = "✅" if finite else "❌"
                print(f"    {status} lines.{col}: min={vals.min():.5f} "
                      f"max={vals.max():.5f}")
                if not finite:
                    issues.append(f"{tag} lines.{col} contains NaN/Inf")
                if col in ["r", "x"] and (vals <= 0).any():
                    warnings.append(f"{tag} lines.{col} has zero/negative values")
            else:
                issues.append(f"{tag} lines.{col} column missing")
                print(f"    ❌ lines.{col}: MISSING")

        # ── 8. Bus–line connectivity (bus0/bus1 in buses.index) ───
        missing_buses = set()
        for col in ["bus0", "bus1"]:
            missing = set(net.lines[col]) - set(net.buses.index)
            missing_buses |= missing
        if missing_buses:
            issues.append(f"{tag} Lines reference buses not in buses.index: "
                         f"{missing_buses}")
        else:
            print(f"\n    ✅ All line endpoints exist in buses.index")

        # ── 9. Simulate one graph build (catches index errors) ────
        print(f"\n  Graph construction smoke test (snapshot 0):")
        try:
            snap = net.snapshots[0]
            bus_to_idx = {b: i for i, b in enumerate(net.buses.index)}
            edges = []
            for _, line in net.lines.iterrows():
                i = bus_to_idx[line.bus0]
                j = bus_to_idx[line.bus1]
                edges += [[i, j], [j, i]]
            edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
            # STSI260404: BUGFIX — align buses_t columns to buses.index order
            x_check = np.column_stack([
                net.buses_t.v_mag_pu.loc[snap, list(net.buses.index)].values,
                net.buses_t.v_ang.loc[snap, list(net.buses.index)].values,
                net.buses_t.p.loc[snap, list(net.buses.index)].values,
                net.buses_t.q.loc[snap, list(net.buses.index)].values,
            ])
            assert np.isfinite(x_check).all(), "Non-finite node features"
            print(f"    ✅ edge_index: {edge_index.shape}, "
                  f"node features: {x_check.shape}, all finite")
        except Exception as e:
            issues.append(f"{tag} Graph construction failed: {e}")
            print(f"    ❌ Graph construction failed: {e}")

    # ── Summary ───────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"SUMMARY for '{name}'")
    if issues:
        print(f"\n❌ {len(issues)} CRITICAL issue(s):")
        for iss in issues:
            print(f"   • {iss}")
    if warnings:
        print(f"\n⚠️  {len(warnings)} warning(s):")
        for w in warnings:
            print(f"   • {w}")
    if not issues and not warnings:
        print("✅ All checks passed — dataset is compatible with PowerFlowDataset")
    elif not issues:
        print("✅ No critical issues — dataset should load correctly")

    print(f"{'='*60}\n")
    return len(issues) == 0




In [ ]:
# =============================================================================
# SECTION 5: GNN MODEL
# =============================================================================

class PowerFlowGNN(nn.Module):
    """
    Graph Attention Network (GATv2) for power flow prediction.
    Predicts per-bus: [vmag, vang, P, Q]
    Predicts per-edge bilinear PTDF: hedges @ W @ hnodes.T
    """

    def __init__(
        self,
        node_features: int = 7,
        edge_features: int = 4,
        hidden_dim: int = 64, # Hiddem dimensions means the size of the feature vectors that are passed between layers in the GNN. A larger hidden_dim allows the model to capture more complex relationships, but also increases computational cost and risk of overfitting. 64 is a common choice for a balance between expressiveness and efficiency.
        num_layers: int = 3,# Number of GNN layers (graph convolutional layers). More layers allow the model to capture more complex interactions between nodes, but can also lead to over-smoothing where node features become too similar. 3 layers is a common choice for many graph tasks.
        heads: int = 4,# Number of attention heads in GAT layers. Multiple heads allow the model to attend to different aspects of the graph structure simultaneously. 4 heads is a common choice that provides a good balance between expressiveness and computational cost.
        dropout: float = 0.0,
        conv_type: str = "gatv2", # Placeholder for potential future extension to other convolution types
        head_mode:str = "standard" # or with new encoder
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.heads = heads
        self.dropout = dropout
        self.conv_type = conv_type
        self.head_mode = head_mode
        # 1. Node embedding
        self.node_embedding = nn.Linear(node_features, hidden_dim)

        # 2. GAT convolution layers
        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(self._make_conv(hidden_dim, hidden_dim, edge_features))

        for _ in range(num_layers):
            self.convs.append(
                GATv2Conv( # GATv2Conv is a graph convolutional layer that uses attention mechanisms to weigh the importance of neighboring nodes when updating node features. It allows for more flexible and powerful message passing compared to traditional GCN layers.
                    hidden_dim, hidden_dim // heads,
                    heads=heads,
                    edge_dim=edge_features,
                    concat=True, # Whether to concatenate the outputs of the attention heads (True) or average them (False). Concatenation allows the model to retain more information from each head, while averaging reduces dimensionality and can help prevent overfitting. True is a common choice for GAT layers.
                    dropout=dropout,# Dropout rate for attention coefficients. A value of 0.0 means no dropout, while a value like 0.1 or 0.2 can help regularize the model and prevent overfitting by randomly dropping some attention connections during training.
                )
            )

        # 3. Output layers for different predictions
        if head_mode=="standard":
            # one prediction head for each of the properties we want to predict
            self.vmag_pred = nn.Linear(hidden_dim, 1)   # Voltage magnitude
            self.vang_pred = nn.Linear(hidden_dim, 1)   # Voltage angle
            self.p_pred = nn.Linear(hidden_dim, 1)      # Active power
            self.q_pred = nn.Linear(hidden_dim, 1)      # Reactive power
        if head_mode=="with_encoder":
            self.pq_head = nn.Linear(hidden_dim, 2)      # for PQ nodes
            self.pv_head = nn.Linear(hidden_dim, 2)      # For PV nodes
            self.slack_head = nn.Linear(hidden_dim, 2)   # For Slack nodes

        # STSI 26.02.11: Edge embedding MLP: (h_src || h_dst || edge_attr) -> hidden_dim
        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * hidden_dim + edge_features, hidden_dim),
            nn.LeakyReLU(),
        )

        # Bilinear PTDF parameter: h_edge^T W h_bus
        self.ptdf_W = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.01)
        
        self._initialize_weights()

    def _make_conv(self, in_dim: int, out_dim: int, edge_features: int):
        if self.conv_type == "gatv2":
            return GATv2Conv(
                in_dim,
                out_dim // self.heads,
                heads=self.heads,
                edge_dim=edge_features,
                concat=True,
                dropout=self.dropout,
            )
        elif self.conv_type == "gcn":
            return GCNConv(in_dim, out_dim, add_self_loops=False)
        elif self.conv_type == "graphconv":
            return GraphConv(in_dim, out_dim)
        else:
            raise ValueError(f"Unknown conv_type: {self.conv_type}")

    def forward(self, data, return_embeddings=False):
        """
        Forward pass: embed -> GAT layers -> node heads + bilinear PTDF.

        Args:
            data: PyG Data/Batch object
            return_embeddings: if True, return (node_pred, h_nodes, h_edges)
                               for use in PTDF loss outside forward
        Returns:
            (node_pred [num_nodes, 4], ptdf_pred [num_edges, num_nodes])
            or (node_pred, h_nodes, h_edges) if return_embeddings=True
        """
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr

        # Initial embedding
        h = self.node_embedding(x)

        # GAT layers
        for conv in self.convs:
            if isinstance(conv, GATv2Conv):
                h = conv(h, edge_index, edge_attr)
            else:
                h = conv(h, edge_index)
            h = F.leaky_relu(h)


        h_nodes = h  # Final node features after all GAT layers

        # Node heads
        # Predictions are made using the output of the last graph convolution layer

        if self.head_mode=="standard":
            vmag_pred = self.vmag_pred(h_nodes)
            vang_pred = self.vang_pred(h_nodes)
            p_pred = self.p_pred(h_nodes)
            q_pred = self.q_pred(h_nodes)
            node_pred = torch.cat([vmag_pred, vang_pred, p_pred, q_pred], dim=-1)  # [num_nodes, 4]
        elif self.head_mode=="with_encoder":
            node_pred = torch.zeros(h_nodes.size(0), 4, device=h_nodes.device)
            pq_mask = data.pq_mask.bool()
            pv_mask = data.pv_mask.bool()
            slack_mask = data.slack_mask.bool()
            
            if pq_mask.any():
                node_pred[pq_mask,0:2]=self.pq_head(h_nodes[pq_mask])  # Predict Vmag, Vang for PQ nodes
            if pv_mask.any():
                pv_out=self.pv_head(h_nodes[pv_mask])  # Predict Vmag, Vang for PV nodes
                node_pred[pv_mask,1]=pv_out[:,0]  # Vmag from PV head
                node_pred[pv_mask,3]=pv_out[:,1]  # Q from PV head
            if slack_mask.any():
                slack_out=self.slack_head(h_nodes[slack_mask])  # Predict Vmag, Vang for Slack nodes
                node_pred[slack_mask, 2] = slack_out[:, 0]  # P from Slack head
                node_pred[slack_mask, 3] = slack_out[:, 1]  # Q from Slack head

        # Edge embeddings
        row, col = data.edge_index
        h_src = h_nodes[row]   # [num_edges, hidden_dim]
        h_dst = h_nodes[col]   # [num_edges, hidden_dim]
        edge_input = torch.cat([h_src, h_dst, edge_attr], dim=-1)  # [num_edges, 2*hidden_dim + edge_features]
        h_edges = self.edge_mlp(edge_input)  # [num_edges, hidden_dim]

        if return_embeddings:
            return node_pred, h_nodes, h_edges

        # Bilinear PTDF: H_W = h_edges @ W,  ptdf_pred = H_W @ h_nodes.T
        H_W = h_edges @ self.ptdf_W                   # [num_edges, hidden_dim]
        ptdf_pred = H_W @ h_nodes.T                   # [num_edges, num_nodes]
        return node_pred, ptdf_pred

    def _initialize_weights(self, gain=1.0):
        """
        STSI 24.09.25: Enhanced weight initialization for the entire network.
        """
        # 1. Initialize node embedding layer (Added 24.09.25)
        nn.init.xavier_uniform_(self.node_embedding.weight, gain=gain)
        nn.init.zeros_(self.node_embedding.bias)

        # 2. Initialize GAT convolution layers (Added 24.09.25)
        for conv in self.convs:
            if hasattr(conv, "lin_l") and conv.lin_l is not None:
                nn.init.xavier_uniform_(conv.lin_l.weight, gain=gain)
            if hasattr(conv, "lin_r") and conv.lin_r is not None:
                nn.init.xavier_uniform_(conv.lin_r.weight, gain=gain)
            if hasattr(conv, "lin_edge") and conv.lin_edge is not None:
                nn.init.xavier_uniform_(conv.lin_edge.weight, gain=gain)
            # GAT layers have multiple linear transformations

        # 3. Initialize output layers
        if self.head_mode == "standard":
            prediction_layers = [
                (self.vmag_pred, 1.0),
                (self.vang_pred, 0.0),
                (self.p_pred,    0.0),
                (self.q_pred,    0.0),
            ]
            for layer, bias_init in prediction_layers:
                nn.init.xavier_uniform_(layer.weight, gain=gain)
                nn.init.constant_(layer.bias, bias_init)
        elif self.head_mode == "with_encoder":
            for layer in (self.pq_head, self.pv_head, self.slack_head):
                nn.init.xavier_uniform_(layer.weight, gain=gain)
                nn.init.zeros_(layer.bias)


# =============================================================================
# SECTION 6: PHYSICS-INFORMED LOSS
# =============================================================================

def compute_power_flow_residual_simple(
    pred, x, Y_matrix,
):
    """
    Debug-friendly AC residual.k

    Uses:
      - True injections P,Q from x at all buses
      - Predicted Vmag,Vang at all buses (ignores PV/slack special cases)
    """
    Y_real, Y_imag = Y_matrix
    device = pred.device

    v_mag = pred[:, 0]
    v_ang = pred[:, 1]

    # angles must be in radians already (dataset uses v_ang in rad)
    v_real = v_mag * torch.cos(v_ang)
    v_imag = v_mag * torch.sin(v_ang)

    I_real = torch.matmul(Y_real, v_real) - torch.matmul(Y_imag, v_imag)
    I_imag = torch.matmul(Y_real, v_imag) + torch.matmul(Y_imag, v_real)

    p_calc = v_real * I_real + v_imag * I_imag
    q_calc = v_imag * I_real - v_real * I_imag

    p_inj = x[:, 3]   # true P (p.u.)
    q_inj = x[:, 4]   # true Q (p.u.)

    p_res = (p_calc - p_inj) ** 2
    q_res = (q_calc - q_inj) ** 2

    return torch.mean(p_res + q_res)


def compute_power_flow_residual_from_pred(
    pred, x, Y_matrix, network, bus_masks=None,
    use_q_partial_mode: bool = False,
):
    """
    Compute AC power-flow residual.

    use_q_partial_mode=False (default/old behaviour):
        All bus types contribute both P and Q residuals.
    use_q_partial_mode=True:
        PV buses contribute only P residual (Q is a free variable for them).
    """
    Y_real, Y_imag = Y_matrix
    n_nodes = pred.size(0)
    device  = pred.device

    if bus_masks is None:
        slack_mask = (x[:, 0] == 1.0)
        pv_mask    = (x[:, 1] == 1.0)
        pq_mask    = (x[:, 2] == 1.0)
    else:
        slack_mask, pv_mask, pq_mask = bus_masks

    # ── Assemble injections ─────────────────────────────────────
    p_inj = torch.zeros(n_nodes, device=device)
    q_inj = torch.zeros(n_nodes, device=device)

    p_inj[pq_mask]    = x[pq_mask, 3]       # known P at PQ buses
    q_inj[pq_mask]    = x[pq_mask, 4]       # known Q at PQ buses
    p_inj[pv_mask]    = x[pv_mask, 3]       # known P at PV buses
    q_inj[pv_mask]    = pred[pv_mask, 3]    # predicted Q at PV buses
    p_inj[slack_mask] = pred[slack_mask, 2] # predicted P at slack
    q_inj[slack_mask] = pred[slack_mask, 3] # predicted Q at slack

    # ── Assemble voltages ────────────────────────────────────────
    v_mag = torch.zeros(n_nodes, device=device)
    v_ang = torch.zeros(n_nodes, device=device)

    v_mag[pq_mask]    = pred[pq_mask, 0]    # predicted Vmag at PQ
    v_ang[pq_mask]    = pred[pq_mask, 1]    # predicted Vang at PQ
    v_mag[pv_mask]    = x[pv_mask, 5]       # known Vmag at PV
    v_ang[pv_mask]    = pred[pv_mask, 1]    # predicted Vang at PV
    v_mag[slack_mask] = x[slack_mask, 5]    # known Vmag at slack
    v_ang[slack_mask] = x[slack_mask, 6]    # known Vang (= 0) at slack

    # ── AC power calculation via Y-bus ──────────────────────────
    v_real = v_mag * torch.cos(v_ang)
    v_imag = v_mag * torch.sin(v_ang)

    I_real = torch.matmul(Y_real, v_real) - torch.matmul(Y_imag, v_imag)
    I_imag = torch.matmul(Y_real, v_imag) + torch.matmul(Y_imag, v_real)

    p_calc = v_real * I_real + v_imag * I_imag
    q_calc = v_imag * I_real - v_real * I_imag

    # ── Residuals ────────────────────────────────────────────────
    p_residual = (p_calc - p_inj) ** 2

    if use_q_partial_mode:
        # exclude PV buses from Q residual — they are Q-free by definition
        q_mask = pq_mask | slack_mask
        q_residual = torch.zeros(n_nodes, device=device)
        q_residual[q_mask] = (q_calc[q_mask] - q_inj[q_mask]) ** 2
    else:
        q_residual = (q_calc - q_inj) ** 2 # include all bus types in the Q residual calculation

    return torch.mean(p_residual + q_residual)


@dataclass
class PhysicsConfig:
    """
    Toggles for every physics guidance term.
    Designed to be swept as a hyperparameter in run_hparam_sweep.
    """
    w_phys: float = 0.0                     # weight on physics loss

    # Variant selection
    physics_mode: str = "rich"            # "simple" or "rich". In simple mode

    # Shared options
    use_power_balance: bool = True          # AC residual on P,Q
    use_angle_ref_penalty: bool = False     # slack angle mean^2

    # Rich-mode options (full PV/slack handling, partial Q)
    use_q_partial_mode: bool = False        # exclude PV buses from Q residual
    
    # PTDF supervision
    use_ptdf_loss: bool = False    # DC PTDF head on/off
    weight_ptdf: float = 0.0      # weight applied to the PTDF loss term
    ptdf_loss_mode: str = "matrix" # "matrix", "flows", "mixed", or "learned_edge" (STSI 26.04.07)
    ptdf_alpha: float = 0.5       # mix ratio for mode="mixed" (matrix share)
    ptdf_branch_mode: str = "lines"  # branches in PTDF: "lines" or "all"

    def label(self) -> str:
        parts = [f"mode-{self.physics_mode}", f"w-{self.w_phys}"]
        if self.use_power_balance:
            parts.append("PB")
        if self.use_angle_ref_penalty:
            parts.append("AR")
        if self.use_q_partial_mode:
            parts.append("Qp")
        if self.use_ptdf_loss:
            ptdf_part = f"PTDF-{self.ptdf_loss_mode}-pt{self.weight_ptdf}-{self.ptdf_branch_mode}"
            if self.ptdf_loss_mode == "mixed":
                ptdf_part += f"-a{self.ptdf_alpha}"
            parts.append(ptdf_part)
        return "_".join(parts)   # e.g. "mode-simple_w-0.0_PB"


def physics_informed_loss_batch(
    pred, target, batch, networks, Y_cache,
    physics_cfg: PhysicsConfig,
    #ptdf_loss: Optional[torch.Tensor] = None, # STSI 260403:Removed as this is no longer computed in this function; it is now computed externally and passed in as an argument to the training loop, which combines it with the physics loss according to PhysicsConfig settings.
):
    """
    Modular physics-informed loss using PhysicsConfig flags.

    total = mse + w_phys * physics_terms

    STSI 26.03.25: MSE is no longer computed here — the training loop passes
    in masked MSE via _masked_mse_loss. This function returns physics loss only.
    The mse_loss return value is kept for API compatibility but is now the
    full (unmasked) MSE for logging purposes only — it is NOT used in total_loss.
    """
    # STSI 26.03.25: full MSE kept for logging/history only — not used in total_loss
    mse_loss = F.mse_loss(pred, target)

    if physics_cfg.w_phys == 0.0:
        zero = torch.tensor(0.0, device=pred.device)
        return mse_loss, mse_loss, zero, zero

    graph_ids  = batch.batch
    num_graphs = int(graph_ids.max().item()) + 1

    total_phys  = 0.0
    total_nodes = 0
    total_angle_ref = 0.0

    for g in range(num_graphs):
        node_mask = (graph_ids == g)
        if not node_mask.any():
            continue

        if batch.network_idx.dim() == 0:
            net_idx = int(batch.network_idx.item())
        else:
            net_idx = int(batch.network_idx[g].item())

        network  = networks[net_idx]
        Y_matrix = Y_cache[net_idx]

        x_g    = batch.x[node_mask]
        pred_g = pred[node_mask]

        # STSI 26.03.25: Fix: — assemble mixed known+predicted state vector
        # per bus type before passing to residual functions, restoring the
        # physically correct formulation from the early implementation.
        # Known values override predictions where the power flow solution
        # provides them as inputs:
        #   PQ  buses: Vmag, Vang predicted;  P, Q    known from x
        #   PV  buses: Vmag, P    known;       Vang, Q predicted
        #   Slack bus: Vmag, Vang known (=ref); P, Q  predicted
        slack_mask_g = batch.slack_mask[node_mask].bool()
        pv_mask_g    = batch.pv_mask[node_mask].bool()
        pq_mask_g    = batch.pq_mask[node_mask].bool()

        # Start from predictions for all variables
        v_mag_g = pred_g[:, 0].clone()
        v_ang_g = pred_g[:, 1].clone()
        p_g     = pred_g[:, 2].clone()
        q_g     = pred_g[:, 3].clone()

        # Override with known values where applicable
        # x layout: [is_slack, is_PV, is_PQ, P, Q, Vmag, Vang]
        v_mag_g[pv_mask_g]    = x_g[pv_mask_g,    5]   # PV:    Vmag known
        p_g[pv_mask_g]        = x_g[pv_mask_g,    3]   # PV:    P known
        p_g[pq_mask_g]        = x_g[pq_mask_g,    3]   # PQ:    P known
        q_g[pq_mask_g]        = x_g[pq_mask_g,    4]   # PQ:    Q known
        v_mag_g[slack_mask_g] = x_g[slack_mask_g, 5]   # Slack: Vmag known
        v_ang_g[slack_mask_g] = x_g[slack_mask_g, 6]   # Slack: Vang=0 known

        # Pack mixed state back into pred-shaped tensor for residual functions
        # Shape: [n_buses_g, 4] = [Vmag, Vang, P, Q]
        pred_g_mixed = torch.stack([v_mag_g, v_ang_g, p_g, q_g], dim=1)

        if physics_cfg.physics_mode == "simple":
            residual = compute_power_flow_residual_simple(
                pred_g_mixed, x_g, Y_matrix,
            )
        else:
            # rich mode: full PV/slack handling, optional partial Q
            bus_masks = (
                slack_mask_g,
                pv_mask_g,
                pq_mask_g,
            )
            residual = compute_power_flow_residual_from_pred(
                pred_g_mixed, x_g, Y_matrix, network,
                bus_masks=bus_masks,
                use_q_partial_mode=physics_cfg.use_q_partial_mode,
            )

        n_g = node_mask.sum().item()
        
            # angle-ref penalty (optional, both modes)
        # ── RESTORED FROM V2.1.2 ──────────────────────────────────────────
        # In V2.1.2 the angle-ref penalty was computed inside the residual
        # function, once per graph, so squaring always happened before any
        # cross-graph aggregation. When PhysicsConfig was introduced (Block 7)
        # this was lifted out to batch level — introducing a cancellation bug.
        # Fix: compute it here inside the per-graph loop, matching V2.1.2.
        angle_ref_g = torch.tensor(0.0, device=pred.device)
        if physics_cfg.use_angle_ref_penalty and slack_mask_g.any():
            angle_ref_g = pred[node_mask][slack_mask_g, 1].mean() ** 2        
        total_angle_ref += angle_ref_g * n_g          #  accumulate separatelyfor logging

        total_phys  += (residual+angle_ref_g) * n_g # include angle_ref in the physics loss for proper weighting and to avoid cross-graph cancellation
        total_nodes += n_g

    physics_loss = total_phys / total_nodes if total_nodes > 0 else torch.tensor(
        0.0, device=pred.device
    )
    angle_ref_loss = total_angle_ref / total_nodes if total_nodes > 0 else torch.tensor(
        0.0, device=pred.device
        )


    # PTDF term (optional) # STSI 260403: Removed from here as it is now added separately during the training loop
    #ptdf_term = torch.tensor(0.0, device=pred.device)
    #if physics_cfg.use_ptdf_loss and ptdf_loss is not None:
    #    ptdf_term = ptdf_loss

    combined_physics = physics_loss #+ ptdf_term

    # STSI 26.03.25: total_loss no longer includes MSE — the training loop
    # adds masked MSE externally. Return only the physics component so the
    # caller can combine: loss = masked_mse + w_phys * physics
    return combined_physics, mse_loss, physics_loss, angle_ref_loss

def get_effective_w_phys(cfg, epoch, warmup_epochs=10):
    if epoch < warmup_epochs:
        return cfg.w_phys * (epoch / warmup_epochs)
    return cfg.w_phys


In [ ]:
# =============================================================================
# SECTION 7: LINE FLOW CALCULATION
# =============================================================================

def calculate_line_flows(network, vmag, vang, t_idx=None):
    """
    Calculate AC line flows from predicted voltages.

    Args:
        network: PyPSA network object
        vmag:    Voltage magnitudes [num_buses], numpy array, p.u.
        vang:    Voltage angles [num_buses], numpy array, radians
        t_idx:   Snapshot index (used for nominal values only)
    Returns:
        dict with keys p0, p1, q0, q1 - each a numpy array of length num_lines (p.u.)
    """
    num_lines = len(network.lines)
    p0 = np.zeros(num_lines)
    p1 = np.zeros(num_lines)
    q0 = np.zeros(num_lines)
    q1 = np.zeros(num_lines)

    all_buses = list(network.buses.index)
    # BUGFIX: use bus_to_idx dict instead of string parsing (topology-safe)
    bus_to_idx = {bus_name: idx for idx, bus_name in enumerate(all_buses)}

    for i, line in enumerate(network.lines.index):
        from_bus = network.lines.loc[line, "bus0"]
        to_bus = network.lines.loc[line, "bus1"]

        # BUGFIX: use dict lookup instead of int(bus.split('-')[1]) - 1
        from_idx = bus_to_idx[from_bus]
        to_idx = bus_to_idx[to_bus]

        r = network.lines.loc[line, "r"]
        x = network.lines.loc[line, "x"]
        b = network.lines.loc[line, "b"] if "b" in network.lines.columns else 0.0

        # Line impedance and admittance
        z = complex(r, x)
        y_series = 1.0 / z if abs(z) > 1e-12 else 0.0
        y_shunt = complex(0, b / 2.0)

        # Complex voltages at from/to buses
        V_from = vmag[from_idx] * np.exp(1j * vang[from_idx])
        V_to = vmag[to_idx] * np.exp(1j * vang[to_idx])

        # Line current from sending end
        I_from = (V_from - V_to) * y_series + V_from * y_shunt
        I_to = (V_to - V_from) * y_series + V_to * y_shunt

        S_from = V_from * np.conj(I_from)
        S_to = V_to * np.conj(I_to)

        s_nom = network.lines.loc[line, "s_nom"]
        # Convert from p.u. to MW/MVAr using system base
        #base_mva = getattr(network, "sn_mva", 100.0)

        p0[i] = S_from.real #* base_mva # removed the base_mva scaling to keep flows in p.u. for better numerical stability and consistency with the rest of the model's inputs/outputs
        q0[i] = S_from.imag #* base_mva
        p1[i] = -S_to.real #* base_mva  # convention: positive = into bus
        q1[i] = -S_to.imag #* base_mva

    return {"p0": p0, "p1": p1, "q0": q0, "q1": q1}


def build_line_results_from_flows(network, flows, snapshot_label):
    """
    Helper to build a line_results-style DataFrame for one snapshot
    from a dict produced by calculate_line_flows.
    """
    idx = pd.Index([snapshot_label], name="snapshot")
    cols = pd.MultiIndex.from_product(
        [network.lines.index, ["P0 pu", "P1 pu", "Q0 pu", "Q1 pu"]]
    )
    df = pd.DataFrame(index=idx, columns=cols, dtype=float)
    for i, line in enumerate(network.lines.index):
        df.loc[snapshot_label, (line, "P0 pu")] = flows["p0"][i]
        df.loc[snapshot_label, (line, "P1 pu")] = flows["p1"][i]
        df.loc[snapshot_label, (line, "Q0 pu")] = flows["q0"][i]
        df.loc[snapshot_label, (line, "Q1 pu")] = flows["q1"][i]
    return df



#=============================================================================
# SECTION 7B: PTDF loss helpers
#=============================================================================

def compute_ptdf_loss_matrix(h_nodes, h_edges, batch, node_batch, edge_batch, num_graphs, model):
    """Helper: compute per-graph PTDF loss from embeddings.
        Uses forward_edge_mask to select one edge per line (avoids double-counting
        the reverse duplicate edges that exist in the undirected graph)."""

    # STSI 26.02.11: PTDF loss per graph in batch (manage changing topologies)
    # STSI 26.03.10: BUGFIX: edge_batch is indexed over all edges; forward_edge_mask selects
    # only the first direction of each line to match y_ptdf shape [n_lines, n_buses]
    forward_mask_full = batch.forward_edge_mask  # [total_edges] bool
    total_ptdf_loss   = 0.0
    graphs_with_edges = 0
    for g in range(num_graphs):
        node_mask = (node_batch == g)
        edge_mask = (edge_batch == g)
        # Apply forward-only filter on top of graph filter
        fwd_edge_mask = edge_mask & forward_mask_full
        if not fwd_edge_mask.any():
            continue
        graphs_with_edges += 1
        h_nodes_g = h_nodes[node_mask]
        h_edges_g = h_edges[fwd_edge_mask]          # [n_lines_g, hidden_dim]
        y_ptdf_g  = batch.y_ptdf_list[g]            # [n_lines_g, n_buses_g]
        H_W_g     = h_edges_g @ model.ptdf_W        # [n_lines_g, hidden_dim]
        ptdf_pred_g = H_W_g @ h_nodes_g.T           # [n_lines_g, n_buses_g]
        total_ptdf_loss += F.mse_loss(ptdf_pred_g, y_ptdf_g)
    if graphs_with_edges > 0:
        return total_ptdf_loss / graphs_with_edges
    return torch.tensor(0.0, device=h_nodes.device)

def compute_ptdf_loss_flows(h_nodes, h_edges, batch, node_batch, edge_batch, num_graphs, model):
    """
    PTDF loss computed in line-flow space:

    For each graph g in the batch:
        - Predict PTDF_g from edge/node embeddings.
        - Build bus injection vector ΔP_g from input features.
        - Compute line flows f_pred_g = PTDF_pred_g @ ΔP_g.
        - Compare to true line flows f_true_g from PyPSA.

    Returns:
        Scalar tensor: mean MSE over graphs with valid lines.
    """
    # 26.03.12 STSI: The original _compute_ptdf_loss computed the loss directly in PTDF space
    # (comparing predicted vs. calculated PTDF matrices). This version computes the loss in
    # line flow space, which is more directly related to physical quantities and may provide
    # a stronger training signal. It requires that the dataset includes true line flow values
    # for each graph, which can be calculated from the original PyPSA networks and included
    # during dataset construction.
    if not hasattr(batch, "forward_edge_mask"):
        raise RuntimeError("Batch missing forward_edge_mask")
    if not hasattr(batch, "y_line_p_list"):
        raise RuntimeError("Batch missing y_line_p_list")
    if not hasattr(batch, "ptdf_line_index_list"):
        raise RuntimeError("Batch missing ptdf_line_index_list")

    forward_mask_full = batch.forward_edge_mask  # [total_edges] bool
    total_loss        = 0.0
    graphs_with_edges = 0

    for g in range(num_graphs):
        # Node and edge masks for this graph
        node_mask     = (node_batch == g)
        edge_mask     = (edge_batch == g)
        fwd_edge_mask = edge_mask & forward_mask_full

        if not fwd_edge_mask.any():
            continue

        # Embeddings for this graph
        h_nodes_g = h_nodes[node_mask]        # [n_buses_g, hidden_dim]
        h_edges_g = h_edges[fwd_edge_mask]    # [n_lines_g, hidden_dim]

        # Predicted PTDF matrix: [n_lines_g, n_buses_g]
        H_W_g       = h_edges_g @ model.ptdf_W
        ptdf_pred_g = H_W_g @ h_nodes_g.T

        # Build bus injection vector ΔP_g from input features x
        # x layout: [is_slack, is_PV, is_PQ, P, Q, Vmag, Vang]
        x_g       = batch.x[node_mask]        # [n_buses_g, feat_dim]
        delta_p_g = x_g[:, 3]                 # use active power injections P as ΔP

        # 26.03.12 STSI: Added line flow loss calculation:
        # True line flows from PyPSA for this graph (must be provided in dataset)
        # Expect shape [n_lines_g]
        # STSI260402: modified to use ptdf_line_index_list to align line flows with PTDF rows, since some graphs may have fewer lines than the max and y_line_p_list is indexed by the max line count.
        # Predicted line flows from PTDF * ΔP
        # [n_lines_g] = [n_lines_g, n_buses_g] @ [n_buses_g]
        f_pred_g = ptdf_pred_g @ delta_p_g

        # True line flows aligned via ptdf_line_index
        line_idx_g = batch.ptdf_line_index_list[g]
        if line_idx_g.numel() == 0:
            continue

        f_true_all_g = batch.y_line_p_list[g]

        max_idx = int(line_idx_g.max().item())
        if max_idx >= len(f_true_all_g):
            raise RuntimeError(
                f"[PTDF flow loss] graph {g}: max line index {max_idx} "
                f">= len(y_line_p_list[{g}]) {len(f_true_all_g)}"
            )

        f_true_g = f_true_all_g[line_idx_g]

        if f_pred_g.shape != f_true_g.shape:
            raise RuntimeError(
                f"[PTDF flow loss] graph {g}: pred shape {tuple(f_pred_g.shape)} "
                f"!= true shape {tuple(f_true_g.shape)}"
            )

        # Flow-based PTDF loss for this graph
        loss_g = F.mse_loss(f_pred_g, f_true_g)
        total_loss        += loss_g
        graphs_with_edges += 1

    if graphs_with_edges > 0:
        return total_loss / graphs_with_edges
    return torch.tensor(0.0, device=h_nodes.device)


# ── STSI 26.04.07: "learned_edge" PTDF mode ─────────────────────────────────
# Learnable PTDF matrices used as (a) edge features for GAT attention and
# (b) unsupervised flow predictors.  One nn.Parameter per network, padded
# to max_buses width so all networks share the same feature dimension.

def initialize_learned_ptdf(networks, max_buses, device=None):
    """Create one learnable PTDF parameter per network.

    Each parameter has shape [n_lines, max_buses]:
      - columns 0..n_buses-1 initialised to 1.0 (active bus columns)
      - columns n_buses..max_buses-1 initialised to 0.0 (padding)

    Returns:
        list[nn.Parameter] indexed by position in *networks*.
    """
    ptdf_params = []
    for net in networks:
        n_lines = len(net.lines)
        n_buses = len(net.buses)
        param = torch.ones(n_lines, max_buses, device=device)
        if n_buses < max_buses:
            param[:, n_buses:] = 0.0
        ptdf_params.append(nn.Parameter(param))
    return ptdf_params


def build_ptdf_augmented_edge_attr(batch, ptdf_params, max_buses, ptdf_offset=0):
    """Concatenate learned PTDF rows to batch.edge_attr for every edge.

    For each edge e in the batch:
      - if ptdf_edge_row_idx[e] >= 0 (supervised forward edge):
            append ptdf_params[net][row_idx]   (shape [max_buses])
      - otherwise: append a zero vector          (shape [max_buses])

    Args:
        batch:        PyG Batch object with .ptdf_edge_row_idx, .network_idx
        ptdf_params:  list[nn.Parameter], one per network (global indexing)
        max_buses:    int, width of every PTDF parameter
        ptdf_offset:  int, added to batch.network_idx to get the global index
                      into ptdf_params  (0 for train, num_train for val, etc.)

    Returns:
        Tensor [total_edges, original_feat + max_buses]
    """
    edge_attr   = batch.edge_attr                     # [E, F]
    total_edges = edge_attr.size(0)
    device      = edge_attr.device

    ptdf_rows = torch.zeros(total_edges, max_buses, device=device)

    node_batch     = batch.batch                       # [N]
    edge_source    = batch.edge_index[0]               # [E]
    edge_batch_ids = node_batch[edge_source]           # [E] graph id per edge
    row_idx        = batch.ptdf_edge_row_idx           # [E] line index or -1

    num_graphs = int(node_batch.max().item()) + 1

    for g in range(num_graphs):
        edge_mask_g = (edge_batch_ids == g)
        net_idx     = int(batch.network_idx[g].item()) + ptdf_offset
        row_idx_g   = row_idx[edge_mask_g]             # [E_g]

        valid = row_idx_g >= 0
        if valid.any():
            valid_line_indices = row_idx_g[valid]       # line row indices
            fetched = ptdf_params[net_idx][valid_line_indices]  # [n_valid, max_buses]

            edge_positions  = torch.where(edge_mask_g)[0]
            valid_positions = edge_positions[valid]
            ptdf_rows[valid_positions] = fetched

    return torch.cat([edge_attr, ptdf_rows], dim=-1)


def compute_ptdf_loss_learned(
    node_pred, batch, node_batch, num_graphs,
    ptdf_params, ptdf_offset=0,
):
    """Unsupervised flow loss: learned_PTDF x predicted_P vs true line flows.

    For each graph g:
      1. Retrieve learned PTDF [n_lines, max_buses], trim to n_buses_g.
      2. Multiply by model-predicted P injections -> predicted line flows.
      3. MSE against ground-truth line flows from PyPSA.

    This provides gradient to both the learned PTDF parameters and the
    model's P-prediction head.

    Args:
        node_pred:    [total_nodes, 4] model output (col 2 = P)
        batch:        PyG Batch with y_line_p_list, ptdf_line_index_list
        node_batch:   [total_nodes] graph assignment
        num_graphs:   int
        ptdf_params:  list[nn.Parameter]
        ptdf_offset:  int, local->global network index offset

    Returns:
        Scalar loss tensor (mean MSE over graphs with lines).
    """
    total_loss        = 0.0
    graphs_with_edges = 0

    for g in range(num_graphs):
        node_mask = (node_batch == g)
        if not node_mask.any():
            continue

        net_idx      = int(batch.network_idx[g].item()) + ptdf_offset
        learned_ptdf = ptdf_params[net_idx]            # [n_lines, max_buses]

        n_buses_g    = int(node_mask.sum().item())
        ptdf_trimmed = learned_ptdf[:, :n_buses_g]     # [n_lines, n_buses_g]

        # Predicted P injections from model output
        pred_p = node_pred[node_mask, 2]               # [n_buses_g]

        # Predicted flows
        f_pred = ptdf_trimmed @ pred_p                 # [n_lines]

        # True flows
        line_idx_g = batch.ptdf_line_index_list[g]
        if line_idx_g.numel() == 0:
            continue

        f_true_all = batch.y_line_p_list[g]
        f_true     = f_true_all[line_idx_g]

        if f_pred.shape != f_true.shape:
            continue

        loss_g = F.mse_loss(f_pred, f_true)
        total_loss        += loss_g
        graphs_with_edges += 1

    if graphs_with_edges > 0:
        return total_loss / graphs_with_edges
    return torch.tensor(0.0, device=node_pred.device)
# ── End STSI 26.04.07 learned_edge functions ─────────────────────────────────


def compute_ptdf_loss(
    h_nodes,
    h_edges,
    batch,
    node_batch,
    edge_batch,
    num_graphs,
    model,
    ptdf_loss_mode: str = "matrix",
    ptdf_alpha: float = 0.5,
    # STSI 26.04.07: additional kwargs for learned_edge mode
    node_pred=None,
    ptdf_params=None,
    ptdf_offset=0,
):
    if ptdf_loss_mode == "matrix":
        return compute_ptdf_loss_matrix(
            h_nodes, h_edges, batch, node_batch, edge_batch, num_graphs, model
        )
    elif ptdf_loss_mode == "flows":
        return compute_ptdf_loss_flows(
            h_nodes, h_edges, batch, node_batch, edge_batch, num_graphs, model
        )
    elif ptdf_loss_mode == "mixed":
        loss_mat = compute_ptdf_loss_matrix(
            h_nodes, h_edges, batch, node_batch, edge_batch, num_graphs, model
        )
        loss_flow = compute_ptdf_loss_flows(
            h_nodes, h_edges, batch, node_batch, edge_batch, num_graphs, model
        )
        return ptdf_alpha * loss_mat + (1.0 - ptdf_alpha) * loss_flow
    elif ptdf_loss_mode == "learned_edge":  # STSI 26.04.07
        return compute_ptdf_loss_learned(
            node_pred, batch, node_batch, num_graphs,
            ptdf_params, ptdf_offset=ptdf_offset,
        )
    else:
        raise ValueError(f"Unknown ptdf_loss_mode: {ptdf_loss_mode!r}")


## Training Loop

In [ ]:

# =============================================================================
# SECTION 8: TRAINING
# =============================================================================
def train_power_flow_gnn(
    networks,
    num_epochs=200,
    batch_size=1,
    lr=0.001,
    conv_type="gatv2",
    hidden_dim=64,
    num_layers=3,
    head_mode: str = "standard",
    physics_cfg: "PhysicsConfig | None" = None,   # ← replaces weight_physics + q_residual_mode
    q_residual_mode="full", 
    use_pnet_balance=True,               # STSI260402: added this argument to enable use of p_net_balance in dataset, which provides a stronger physical signal to the model and is needed for the PTDF loss. Kept as an argument to allow ablation of this feature if desired.
    weight_ptdf=0.1,                 # STSI260402: removed and included in PhysicsConfig
    ptdf_loss_mode: str = "matrix",  # "matrix", "flows", or "mixed" STSI260402: removed and included in PhysicsConfig 
    ptdf_alpha: float = 0.5,         # only used if mode == "mixed" # STSI260402: removed and included in PhysicsConfig
    use_edge_features=True,
    max_n_test=None,
    # DISTSLACK: propagate distribute_slack flag to dataset construction
    distribute_slack=False,
    y_matrix_source: str = "auto",   # STSI260320: controls Y source for physics loss
    warmup_epochs: int = 10,           # STSI 26.03.25: exposed for sweep control
    seed=42,
    return_debug_objects: bool = False,# STSI260402 added for enabling return of extra debug info without affecting main return values
    tqdm_file=None,  # file handle for tqdm output (e.g. real stderr when verbose=False)
):
    """
    Train power flow GNN with proper network-based train/val/test split.
    Extended version that returns per-epoch histories and aggregated test metrics.

    DISTSLACK: pass distribute_slack=True to enable distributed slack handling
               in the physics loss and masking.

    Returns:
        model, history (dict of per-epoch lists), test_metrics (dict)
    """
    if physics_cfg is None:
        physics_cfg = PhysicsConfig(w_phys=0.0)

    # STSI 16.02.26: Extended train_power_flow_gnn to return history & test metrics
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    networks_copy = networks.copy()
    random.shuffle(networks_copy)
    num_train_networks = int(0.7 * len(networks_copy))
    num_val_networks   = int(0.85 * len(networks_copy)) - num_train_networks
    train_networks = networks_copy[:num_train_networks]
    val_networks   = networks_copy[num_train_networks:num_train_networks + num_val_networks]
    test_networks  = networks_copy[num_train_networks + num_val_networks:]

    # Debug: Verify split
    print(f"Network split: Train={len(train_networks)}, Val={len(val_networks)}, Test={len(test_networks)}")
    print(f"Total {len(train_networks)+len(val_networks)+len(test_networks)} of {len(networks)}")

    # Create datasets from split networks
    train_dataset = PowerFlowDataset(
        train_networks,
        use_edge_features=use_edge_features,
        use_pnet_balance=use_pnet_balance,  # STSI260402: added this as argument to enable p_net_balance in dataset for all splits, which is needed for the PTDF loss and also provides a stronger physical signal to the model across the board.
        ptdf_branch_mode=physics_cfg.ptdf_branch_mode, # STSI260402: pass PTDF branch mode to dataset for all splits, since it controls whether the dataset includes PTDF targets for all branches or just lines, which affects the shape of the PTDF loss and is needed for consistency across training/validation/test.
    )
    val_dataset = PowerFlowDataset(
        val_networks,
        use_edge_features=use_edge_features,
        use_pnet_balance=use_pnet_balance,  # STSI260402: added this as argument to enable p_net_balance in dataset for all splits, which is needed for the PTDF loss and also provides a stronger physical signal to the model across the board.
        ptdf_branch_mode=physics_cfg.ptdf_branch_mode, # STSI260402: pass PTDF branch mode to dataset for all splits, since it controls whether the dataset includes PTDF targets for all branches or just lines, which affects the shape of the PTDF loss and is needed for consistency across training/validation/test.
    )
    test_dataset = PowerFlowDataset(
        test_networks,
        use_edge_features=use_edge_features,
        use_pnet_balance=use_pnet_balance,  # STSI260402: added this as argument to enable p_net_balance in dataset for all splits, which is needed for the PTDF loss and also provides a stronger physical signal to the model across the board.
        ptdf_branch_mode=physics_cfg.ptdf_branch_mode, # STSI260402: pass PTDF branch mode to dataset for all splits, since it controls whether the dataset includes PTDF targets for all branches or just lines, which affects the shape of the PTDF loss and is needed for consistency across training/validation/test.
    )

    # Debug: Verify dataset sizes
    print(f"Dataset sizes: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}")
    sample = train_dataset[0]
    print(f"Feature dim: {sample.x.size(1)}")        # must be 8
    print(f"p_net_balance: {sample.p_net_balance}")   # must be non-zero

        # ── Dataset diagnostic (compare with v2.1.2 output) ──────────────────
    _diag = train_dataset[0]
    _sm = _diag.slack_mask.bool()
    _pvm = _diag.pv_mask.bool()
    _pqm = _diag.pq_mask.bool()
    print(f"\n--- DATASET DIAGNOSTIC (sample 0) ---")
    print(f"x shape: {_diag.x.shape}  y shape: {_diag.y.shape}")
    print(f"edge_index shape: {_diag.edge_index.shape}  ({_diag.edge_index.shape[1]} edges)")
    print(f"bus counts: slack={_sm.sum()}, PV={_pvm.sum()}, PQ={_pqm.sum()}")
    if _sm.any():
        print(f"Slack x[6] (Vang, should be ~0): {_diag.x[_sm, 6]}")
        print(f"Slack x[5] (Vmag):               {_diag.x[_sm, 5]}")
        print(f"Slack x[3:5] (P,Q, should be 0): {_diag.x[_sm, 3:5]}")
        print(f"Slack y (target):                {_diag.y[_sm]}")
    if _pvm.any():
        print(f"PV x[5] (Vmag, should be ~1.0):  {_diag.x[_pvm, 5][:3]}")
        print(f"PV x[6] (Vang, should be 0):     {_diag.x[_pvm, 6][:3]}")
        print(f"PV x[3] (P, known):              {_diag.x[_pvm, 3][:3]}")
        print(f"PV x[4] (Q, should be 0):        {_diag.x[_pvm, 4][:3]}")
        print(f"PV y (target):                   {_diag.y[_pvm][:3]}")
    if _pqm.any():
        print(f"PQ x[5:7] (Vmag,Vang, should be 0): {_diag.x[_pqm, 5:7][:3]}")
        print(f"PQ x[3:5] (P,Q, known):             {_diag.x[_pqm, 3:5][:3]}")
        print(f"PQ y (target):                       {_diag.y[_pqm][:3]}")
    if hasattr(_diag, 'edge_attr'):
        print(f"edge_attr shape: {_diag.edge_attr.shape}")
        print(f"edge_attr sample (first edge): {_diag.edge_attr[0]}")
    print(f"--- END DIAGNOSTIC ---\n")


    # Cache Y-matrices to reduce computation during training
    logger.info("Precomputing admittance matrices...")
    train_Y_cache = precompute_Y_matrices(train_networks, y_matrix_source=y_matrix_source)  # STSI260320: y_matrix_source added to enable choice between using pypsa retrieved or manually calculated Y matrix in physics losses
    val_Y_cache   = precompute_Y_matrices(val_networks,   y_matrix_source=y_matrix_source)  # STSI260320: y_matrix_source added to enable choice between using pypsa retrieved or manually calculated Y matrix in physics losses
    test_Y_cache  = precompute_Y_matrices(test_networks,  y_matrix_source=y_matrix_source)  # STSI260320: y_matrix_source added to enable choice between using pypsa retrieved or manually calculated Y matrix in physics losses
    logger.info("Y-matrices cached successfully")

    # ── STSI 26.04.07: learned_edge PTDF initialisation ─────────────────
    _is_learned_edge = (physics_cfg.ptdf_loss_mode == "learned_edge"
                        and physics_cfg.use_ptdf_loss
                        and physics_cfg.weight_ptdf > 0.0)
    ptdf_params = None
    max_buses = 0
    train_ptdf_offset = 0
    val_ptdf_offset = num_train_networks
    test_ptdf_offset = num_train_networks + len(val_networks)
    if _is_learned_edge:
        max_buses = max(len(net.buses) for net in networks_copy)
        ptdf_params = initialize_learned_ptdf(networks_copy, max_buses)
        print(f"[learned_edge] Initialized {len(ptdf_params)} PTDF params, max_buses={max_buses}")

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        collate_fn=collate_with_ptdf
    )  # STSI 26.02.10: variable graph sizes + PTDF
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=collate_with_ptdf
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=collate_with_ptdf
    )

    # Initialize model based on sample
    sample_data = train_dataset[0]


    # STSI 26.04.07: augment edge feature dim when learned_edge PTDF rows are appended
    _edge_feat_dim = sample_data.edge_attr.size(1) + (max_buses if _is_learned_edge else 0)

    model = PowerFlowGNN(
        node_features=sample_data.x.size(1),
        edge_features=_edge_feat_dim,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        conv_type=conv_type,
        head_mode=head_mode,
    )
    print(model)
    #check_model_input_dim(model, train_dataset, label="train") # removed for now as it caused error and was not really needed now...
    # Optimizer & scheduler
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=10, verbose=True
    )

    # STSI 26.04.07: add learned PTDF params to optimizer
    if ptdf_params is not None:
        optimizer.add_param_group({'params': ptdf_params})

    # STSI 16.02.26: Extended tracking of per-epoch components
    train_total_list, val_total_list = [], []
    train_mse_list, train_phys_list, train_ptdf_list = [], [], []
    val_mse_list,   val_phys_list,   val_ptdf_list   = [], [], []
    train_angle_ref_list, val_angle_ref_list = [], []
    best_val_loss   = float("inf")
    best_state_dict = None  # 26.03.12 STSI: track best model weights in memory

    # STSI 26.03.26: Linear warm-up for w_phys to avoid early collapse to average solution
    import dataclasses

    def _effective_physics_cfg(base_cfg: PhysicsConfig, epoch: int, warmup_epochs: int = 10):
        """
        Linearly ramp w_phys from 0 → base_cfg.w_phys over warmup_epochs.
        After warmup, keep w_phys constant.
        Returns a shallow copy of base_cfg with adjusted w_phys.
        """
        if base_cfg.w_phys <= 0.0:
            return base_cfg
        if epoch >= warmup_epochs:
            return base_cfg
        scale = float(epoch + 1) / float(warmup_epochs)   # epoch 0 → 1/warmup, epoch warmup-1 → ~1
        return dataclasses.replace(base_cfg, w_phys=base_cfg.w_phys * scale)

    # STSI 26.03.25: Restored masked MSE from early implementation — only penalise
    # unknown variables per bus type to avoid gradient dilution and shared-head conflicts.
    # PQ  → predict Vmag (0), Vang (1)
    # PV  → predict Vang (1), Q (3)
    # Slack → predict P (2), Q (3)

    def _masked_mse_loss(
        node_pred: torch.Tensor,
        target: torch.Tensor,
        batch,
        debug: bool = False,
    ) -> torch.Tensor:
        """
        MSE loss restricted to physically unknown variables per bus type.

        Bus type  | Predict (unknown)    | Skip (known input)
        ----------|----------------------|--------------------
        PQ        | Vmag (0), Vang (1)   | P (3), Q (4)
        PV        | Vang (1), Q (3)      | Vmag (5), P (3)
        Slack     | P (2), Q (3)         | Vmag (5), Vang (6)

        Reads masks from batch.slack_mask / batch.pv_mask / batch.pq_mask
        (bool tensors, node-level, set by PowerFlowDataset and propagated by
        collate_with_ptdf via Batch.from_data_list auto-concatenation).

        Returns 0 if all masks are missing (with a warning) to fail loudly.
        """
        # ── Source masks from batch attributes (preferred) or x columns ──────────
        if hasattr(batch, "slack_mask") and hasattr(batch, "pv_mask") and hasattr(batch, "pq_mask"):
            slack_mask = batch.slack_mask.bool()
            pv_mask    = batch.pv_mask.bool()
            pq_mask    = batch.pq_mask.bool()
            mask_source = "batch_attr"
        else:
            # Fallback: derive from x feature columns [is_slack=0, is_PV=1, is_PQ=2]
            logger.warning(
                "_masked_mse_loss: batch.slack_mask/pv_mask/pq_mask not found. "
                "Falling back to x[:,0:3] — check collate_with_ptdf and PowerFlowDataset."
            )
            slack_mask = batch.x[:, 0].bool()
            pv_mask    = batch.x[:, 1].bool()
            pq_mask    = batch.x[:, 2].bool()
            mask_source = "x_cols"

        if debug:
            print(f"[_masked_mse_loss] mask_source={mask_source}")
            print(f"  slack_mask sum={slack_mask.sum().item()}, "
                f"pv_mask sum={pv_mask.sum().item()}, "
                f"pq_mask sum={pq_mask.sum().item()}")

        loss = torch.tensor(0.0, device=node_pred.device)
        n    = 0

        # PQ buses: predict Vmag (col 0) and Vang (col 1)
        if pq_mask.any():
            loss = loss + F.mse_loss(node_pred[pq_mask][:, 0:2], target[pq_mask][:, 0:2])
            n += 1

        # PV buses: predict Vang (col 1) and Q (col 3)
        if pv_mask.any():
            pv_cols = torch.tensor([1, 3], device=node_pred.device)
            loss = loss + F.mse_loss(
                node_pred[pv_mask][:, pv_cols],
                target[pv_mask][:, pv_cols],
            )
            n += 1

        # Slack buses: predict P (col 2) and Q (col 3)
        if slack_mask.any():
            loss = loss + F.mse_loss(node_pred[slack_mask][:, 2:4], target[slack_mask][:, 2:4])
            n += 1

        if n == 0:
            logger.warning(
                "_masked_mse_loss: all masks are empty — returning 0. "
                "Training gradients will not flow. Check mask propagation."
            )
            return torch.tensor(0.0, device=node_pred.device, requires_grad=True)

        return loss / max(n, 1)
# STSI260402  removed nested helpers for ptdf losses (_compute_ptdf_loss, _compute_ptdf_loss_flows, _compute_ptdf_loss_matrix) to expose these to unit testing and smoke test
# ── Move _epoch0_debug definition here — BEFORE the epoch loop ──────────

    def _epoch0_debug(batch, node_pred):
        print("\n" + "="*60)
        print("EPOCH 0 VERIFICATION CHECKS")
        print("="*60)

        print("\n[P1] Mask propagation through collate_with_ptdf:")
        for attr in ("slack_mask", "pv_mask", "pq_mask"):
            if hasattr(batch, attr):
                s = getattr(batch, attr).sum().item()
                print(f"  [OK] batch.{attr} present -- sum={int(s)}")
            else:
                print(f"  [FAIL] batch.{attr} MISSING -- _masked_mse_loss will return 0!")

        print("\n[P2] Masked vs full MSE:")
        full_mse   = F.mse_loss(node_pred, batch.y)
        masked_mse = _masked_mse_loss(node_pred, batch.y, batch)
        print(f"  Full MSE:   {full_mse.item():.6f}")
        print(f"  Masked MSE: {masked_mse.item():.6f}")
        if abs(full_mse.item() - masked_mse.item()) < 1e-8:
            print("  [WARN] Full MSE == Masked MSE -- masks may be inactive!")
        else:
            print("  [OK] Masked MSE differs from full MSE -- masks are active.")

        print("\n[P3] Physics state assembly:")
        slack_mask = batch.slack_mask.bool()
        pv_mask    = batch.pv_mask.bool()
        if slack_mask.any():
            print(f"  Slack P predicted: {node_pred[slack_mask, 2].detach()[:3]}")
            print(f"  Slack Vang input (should be ~0): {batch.x[slack_mask, 6].detach()[:3]}")
        else:
            print("  [WARN] No slack buses found in this batch!")
        if pv_mask.any():
            print(f"  PV Vmag input (should be ~1.0-1.05): {batch.x[pv_mask, 5].detach()[:3]}")

        print(f"\n[Info] batch.x shape: {batch.x.shape}  (expect [N,7] or [N,8])")
        print("="*60 + "\n")

    # ── Move flag here — BEFORE the epoch loop ───────────────────────────────
    _first_batch_debug_done = False
    print(f"train_loader length: {len(train_loader)}")
    print(f"train_dataset length: {len(train_dataset)}")
    print(f"train_networks length: {len(train_networks)}") 
    _tqdm_kw = dict(desc="Training", unit="epoch")
    if tqdm_file is not None:
        _tqdm_kw["file"] = tqdm_file
    for epoch in tqdm(range(num_epochs), **_tqdm_kw):

        # STSI 26.03.23: apply linear warm-up to w_phys during training
        physics_cfg_epoch = _effective_physics_cfg(physics_cfg, epoch, warmup_epochs=warmup_epochs)

        # ---- Training ----
        model.train()
        epoch_loss = epoch_mse = epoch_physics = epoch_angle_ref = epoch_ptdf = 0.0

        for batch in train_loader:
            optimizer.zero_grad()

            # STSI 26.04.07: augment edge_attr with learned PTDF rows
            if ptdf_params is not None:
                batch.edge_attr = build_ptdf_augmented_edge_attr(
                    batch, ptdf_params, max_buses, ptdf_offset=train_ptdf_offset)

            node_pred, h_nodes, h_edges = model(batch, return_embeddings=True)

            #=====================Debug block
            if epoch == 0 and not _first_batch_debug_done:
                _epoch0_debug(batch, node_pred)
                _first_batch_debug_done = True
            #=====================End Debug block
            # ── MSE loss — masked to unknown variables per bus type ──────────
            # STSI 26.03.25: replaced full F.mse_loss(node_pred, batch.y) with
            # _masked_mse_loss to avoid gradient dilution from trivially-known
            # outputs (e.g. slack Vmag/Vang, PQ P/Q) and shared-head conflicts.
            mse = _masked_mse_loss(node_pred, batch.y, batch)

            # ── Physics-informed node loss ───────────────────────────────────
            if physics_cfg_epoch.w_phys > 0.0:
                # STSI 26.03.25: physics_informed_loss_batch must internally use
                # the mixed known+predicted state vector (see Fix 2 in physics fn)
                loss_nodes, _, physics, angle_ref = physics_informed_loss_batch( 
                    node_pred, batch.y, batch,
                    networks=train_networks,
                    Y_cache=train_Y_cache,
                    physics_cfg=physics_cfg_epoch,
                )
                # STSI 26.03.25: combine masked MSE with physics loss explicitly
                # so that mse tracked in history is always the masked variant
                loss_nodes = mse + physics_cfg_epoch.w_phys * physics
            else:
                loss_nodes = mse
                physics    = torch.tensor(0.0, device=node_pred.device)
                angle_ref  = torch.tensor(0.0, device=node_pred.device) #STSI 260403: added to log angle reference error even when physics loss is disabled

            # ── PTDF loss ────────────────────────────────────────────────────
            node_batch = batch.batch
            row, _     = batch.edge_index
            edge_batch = node_batch[row]
            num_graphs = int(node_batch.max().item()) + 1

            if physics_cfg.weight_ptdf > 0.0 and physics_cfg.use_ptdf_loss:
                ptdf_loss = compute_ptdf_loss(
                    h_nodes, h_edges, batch, node_batch, edge_batch, num_graphs, model,
                    ptdf_loss_mode=physics_cfg.ptdf_loss_mode,
                    ptdf_alpha=physics_cfg.ptdf_alpha,
                    node_pred=node_pred, ptdf_params=ptdf_params,  # STSI 26.04.07
                    ptdf_offset=train_ptdf_offset,                 # STSI 26.04.07
                )
            else:
                ptdf_loss = torch.tensor(0.0, device=node_pred.device)

            loss = loss_nodes + physics_cfg.weight_ptdf * ptdf_loss

            loss.backward()
            optimizer.step()

            epoch_loss    += loss.item()
            epoch_mse     += mse.item()
            epoch_physics += physics.item()
            epoch_angle_ref += angle_ref.item() # STSI 260403: added to log angle reference error separately
            epoch_ptdf    += ptdf_loss.item()

        # Backward step averages
        train_loss = epoch_loss / len(train_loader)
        train_total_list.append(train_loss)
        train_mse_list.append(epoch_mse      / len(train_loader))
        train_phys_list.append(epoch_physics / len(train_loader))
        train_angle_ref_list.append(epoch_angle_ref / len(train_loader)) # STSI 260403: log angle reference error separately
        train_ptdf_list.append(epoch_ptdf    / len(train_loader))

        # ---- Validation ----
        model.eval()
        val_loss = val_mse = val_physics = val_angle_ref = val_ptdf = 0.0
        with torch.no_grad():
            for batch in val_loader:
                # STSI 26.04.07: augment edge_attr with learned PTDF rows
                if ptdf_params is not None:
                    batch.edge_attr = build_ptdf_augmented_edge_attr(
                        batch, ptdf_params, max_buses, ptdf_offset=val_ptdf_offset)

                node_pred, h_nodes, h_edges = model(batch, return_embeddings=True)

                # STSI 26.03.25: use masked MSE for validation consistency
                mse = _masked_mse_loss(node_pred, batch.y, batch)

                if physics_cfg.w_phys > 0.0:
                    _, _, physics, angle_ref = physics_informed_loss_batch(
                        node_pred, batch.y, batch,
                        networks=val_networks,
                        Y_cache=val_Y_cache,
                        physics_cfg=physics_cfg,   # use full configured weight for eval
                    )
                    loss_nodes = mse + physics_cfg.w_phys * physics
                else:
                    loss_nodes = mse
                    physics    = torch.tensor(0.0, device=node_pred.device) # STSI 260403: Physics losses logged separately but includes both power residual and angle reference components
                    angle_ref  = torch.tensor(0.0, device=node_pred.device) # STSI 260403: added to log angle reference error even when physics loss is disabled

                node_batch = batch.batch
                row, _     = batch.edge_index
                edge_batch = node_batch[row]
                num_graphs = int(node_batch.max().item()) + 1

                if physics_cfg.weight_ptdf > 0.0 and physics_cfg.use_ptdf_loss:
                    #node_batch = batch.batch
                    #row, _ = batch.edge_index
                    #edge_batch = node_batch[row]
                    #num_graphs = batch.num_graphs

                    ptdf_loss = compute_ptdf_loss(
                        h_nodes, h_edges, batch,
                        node_batch, edge_batch, num_graphs,
                        model=model,
                        ptdf_loss_mode=getattr(physics_cfg, "ptdf_loss_mode", "matrix"),
                        ptdf_alpha=getattr(physics_cfg, "ptdf_alpha", 0.5),
                        node_pred=node_pred, ptdf_params=ptdf_params,  # STSI 26.04.07
                        ptdf_offset=val_ptdf_offset,                   # STSI 26.04.07
                    )
                else:
                    ptdf_loss = torch.tensor(0.0, device=node_pred.device)

                loss        = loss_nodes + physics_cfg.weight_ptdf * ptdf_loss
                val_loss    += loss.item()
                val_mse     += mse.item()
                val_physics += physics.item()
                val_angle_ref += angle_ref.item()
                val_ptdf    += ptdf_loss.item()

        val_loss /= len(val_loader)
        val_total_list.append(val_loss)
        val_mse_list.append(val_mse       / len(val_loader))
        val_phys_list.append(val_physics  / len(val_loader))
        val_angle_ref_list.append(val_angle_ref / len(val_loader)) # STSI 260403: log angle reference error separately
        val_ptdf_list.append(val_ptdf     / len(val_loader))

        scheduler.step(val_loss)

        # Save best model — use physics label in filename
        save_name = (
            f"training_results_saved/best_model"
            f"_{physics_cfg.label()}"
            f"_pt{physics_cfg.weight_ptdf}_bs{batch_size}_lr{lr}.pt"
        )

        if val_loss < best_val_loss:
            best_val_loss   = val_loss
            best_state_dict = copy.deepcopy(model.state_dict())
            safe_torch_save(model.state_dict(), save_name)

        print(
            f"Epoch {epoch+1}/{num_epochs}, "
            f"Train Loss {train_loss:.6f}, "
            f"Val Loss {val_loss:.6f}, "
            f"LR {optimizer.param_groups[0]['lr']:.6f}"
        )
        # Debug to detect data leakage (epoch 0 only)
        if epoch == 0:
            print("DATA LEAKAGE CHECK")
            print(f"Input features batch.x shape {batch.x.shape}")
            print(f"Input features sample: {batch.x[:5]}")
            print(f"Target features batch.y shape {batch.y.shape}")
            print(f"Target features sample: {batch.y[:5]}")
            print("CHECKING FOR LEAKAGE")
            pq_indices = torch.where(batch.pq_mask)[0]
            if len(pq_indices) > 0:
                print(f"PQ bus indices: {pq_indices}")
                print(f"Input at PQ bus 0: {batch.x[pq_indices[0]]}")
                print(f"Target at PQ bus 0: {batch.y[pq_indices[0]]}")

    # ---- Test evaluation ----
    # after training loop, once you've tracked best_state_dict
    # 26.03.12 STSI: ensure we evaluate the best model, not the last epoch
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
    model.eval()

    test_total = test_mse = test_physics = test_angle_ref = test_ptdf = 0.0
    with torch.no_grad():
        for batch in test_loader:
            # STSI 26.04.07: augment edge_attr with learned PTDF rows
            if ptdf_params is not None:
                batch.edge_attr = build_ptdf_augmented_edge_attr(
                    batch, ptdf_params, max_buses, ptdf_offset=test_ptdf_offset)

            node_pred, h_nodes, h_edges = model(batch, return_embeddings=True)
            mse = _masked_mse_loss(node_pred, batch.y, batch)
            # STSI 26.03.25: use masked MSE for test consistency
            mse = _masked_mse_loss(node_pred, batch.y, batch)

            if physics_cfg.w_phys > 0.0:
                _, _, physics, angle_ref = physics_informed_loss_batch(
                    node_pred, batch.y, batch,
                    networks=test_networks,
                    Y_cache=test_Y_cache,
                    physics_cfg=physics_cfg,
                )
                loss_nodes = mse + physics_cfg.w_phys * physics
            else:
                loss_nodes = mse
                physics    = torch.tensor(0.0, device=node_pred.device)
                angle_ref  = torch.tensor(0.0, device=node_pred.device) # STSI 260403: added to log angle reference error separately but includes both power residual and angle reference components

            node_batch = batch.batch
            row, _     = batch.edge_index
            edge_batch = node_batch[row]
            num_graphs = int(node_batch.max().item()) + 1

            if physics_cfg.weight_ptdf > 0.0 and physics_cfg.use_ptdf_loss:
                    ptdf_loss = compute_ptdf_loss(
                        h_nodes,
                        h_edges,
                        batch,
                        node_batch,
                        edge_batch,
                        num_graphs,
                        model,
                        ptdf_loss_mode=physics_cfg.ptdf_loss_mode,
                        ptdf_alpha=physics_cfg.ptdf_alpha,
                        node_pred=node_pred, ptdf_params=ptdf_params,  # STSI 26.04.07
                        ptdf_offset=test_ptdf_offset,                  # STSI 26.04.07
                    )
            else:
                ptdf_loss = torch.tensor(0.0, device=node_pred.device)

            loss         = loss_nodes + physics_cfg.weight_ptdf * ptdf_loss
            test_total   += loss.item()
            test_mse     += mse.item()
            test_physics += physics.item() # physics loss logged but includes both power residual and angle reference components
            test_angle_ref += angle_ref.item() # STSI 260403: log angle reference error separately 
            test_ptdf    += ptdf_loss.item()

    test_total   /= len(test_loader)
    test_mse     /= len(test_loader)
    test_physics /= len(test_loader)
    test_angle_ref /= len(test_loader) # STSI 260403: log angle reference error separately
    test_ptdf    /= len(test_loader)

    print("TEST METRICS")
    print(f"Test total loss: {test_total:.6f}")
    print(f"Test MSE loss:   {test_mse:.6f}")
    print(f"Test physics:    {test_physics:.6f}")
    print(f"Test angle ref:  {test_angle_ref:.6f}") # STSI 260403: log angle reference error separately
    print(f"Test PTDF:       {test_ptdf:.6f}")

    history = {
        "train_total": train_total_list, "val_total": val_total_list,
        "train_mse":   train_mse_list,   "val_mse":   val_mse_list,
        "train_phys":  train_phys_list,  "val_phys":  val_phys_list,
        "train_angle_ref": train_angle_ref_list,"val_angle_ref":   val_angle_ref_list,    # STSI 260403: added angle reference error to history tracking
        "train_ptdf":  train_ptdf_list,  "val_ptdf":  val_ptdf_list,
    }

    test_metrics = {
        "test_total":   test_total,
        "test_mse":     test_mse,
        "test_physics": test_physics,
        "test_angle_ref": test_angle_ref, # STSI 260403: added angle reference error to test metrics
        "test_ptdf":    test_ptdf,
    }

    # Optional: add node-wise metrics over all test networks
    nodewise = evaluate_gnn_on_test_set(
        model, test_networks, use_edge_features=use_edge_features,
        ptdf_params=ptdf_params, max_buses=max_buses,       # STSI 26.04.07
        ptdf_offset=test_ptdf_offset,                       # STSI 26.04.07
    )
    test_metrics.update(nodewise)

    # STSI 26.04.07: store learned_edge artefacts so callers can augment edge_attr
    test_metrics["_ptdf_params"] = ptdf_params   # list[nn.Parameter] or None
    test_metrics["_max_buses"]   = max_buses      # int (0 when not learned_edge)

    if return_debug_objects: #STSI260402: return debug objects if flag is set
        debug_objects = {
            "train_networks": train_networks,
            "val_networks": val_networks,
            "test_networks": test_networks,
            "train_dataset": train_dataset,
            "val_dataset": val_dataset,
            "test_dataset": test_dataset,
            "train_loader": train_loader,
            "val_loader": val_loader,
            "test_loader": test_loader,
            "train_Y_cache": train_Y_cache,
            "val_Y_cache": val_Y_cache,
            "test_Y_cache": test_Y_cache,
        }
        return model, history, test_metrics, debug_objects

    return model, history, test_metrics

def safe_torch_save(state_dict, path, max_retries=5, delay=0.5):
    import time, os, torch
    tmp_path = f"{path}.tmp_{os.getpid()}"
    for attempt in range(1, max_retries + 1):
        try:
            torch.save(state_dict, tmp_path)
            if os.path.exists(path):
                os.remove(path)
            os.replace(tmp_path, path)
            return
        except Exception as e:
            if attempt == max_retries:
                print(f"[WARN] Failed to save model to {path} after {max_retries} attempts: {e}")
                try:
                    if os.path.exists(tmp_path):
                        os.remove(tmp_path)
                except Exception:
                    pass
                return
            time.sleep(delay)

#debug helpers
# %%
# Replace _epoch0_debug with ASCII-safe version (Windows cp1252 compatible)

def check_model_input_dim(model, dataset, label="train"):
    """Verify model input dim matches dataset node feature dim."""
    sample = dataset[0]
    expected_dim = sample.x.size(1)

    # GATv2Conv projects inputs via a linear layer before attention.
    # The correct place to read input dim is the weight matrix of that
    # linear layer, not in_channels (which stores hidden/output dim).
    model_in_dim = None
    try:
        # PowerFlowGNN typically has an input projection: self.input_proj or
        # the first conv layer's lin_src / lin_l weight
        first_conv = model.convs[0]
        if hasattr(first_conv, "lin_l"):
            model_in_dim = first_conv.lin_l.weight.size(1)
        elif hasattr(first_conv, "lin_src"):
            model_in_dim = first_conv.lin_src.weight.size(1)
        elif hasattr(model, "input_proj"):
            model_in_dim = model.input_proj.weight.size(1)
    except Exception:
        model_in_dim = None

    if model_in_dim is None:
        print(f"⚠️  [{label}] Could not determine model input dim — skipping check. "
              f"Dataset node feature dim = {expected_dim}.")
        return

    if model_in_dim != expected_dim:
        raise ValueError(
            f"Model input dim ({model_in_dim}) != dataset node feature dim ({expected_dim}). "
            f"Rebuild model with node_features=sample_data.x.size(1)."
        )

    print(f"✅ [{label}] Model input dim {model_in_dim} matches dataset dim {expected_dim}.")

In [ ]:
# =============================================================================
# SECTION 9: INFERENCE & PREDICTION
# =============================================================================

def _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx, ptdf_offset=0, device=None):
    """Augment data.edge_attr with learned PTDF rows for learned_edge models.

    Shared helper used by evaluate_model, compare_pf_conventional_vs_gnn,
    predict_network_results_with_masks, and evaluate_gnn_on_test_set.

    Args:
        data:          single PyG Data object (already moved to device)
        ptdf_params:   list[nn.Parameter] or None
        max_buses:     int, PTDF parameter width
        net_idx:       int, network index within the dataset
        ptdf_offset:   int, added to net_idx to get global ptdf_params index
        device:        torch device (inferred from data.edge_attr if None)

    Returns:
        data with edge_attr augmented in-place. No-op if ptdf_params is None.
    """
    if ptdf_params is None or max_buses <= 0:
        return data
    if device is None:
        device = data.edge_attr.device
    row_idx = data.ptdf_edge_row_idx  # [E] line index or -1
    ptdf_rows = torch.zeros(data.edge_attr.size(0), max_buses, device=device)
    valid = (row_idx >= 0) & (row_idx < ptdf_params[net_idx + ptdf_offset].size(0))
    if valid.any():
        ptdf_rows[valid] = ptdf_params[net_idx + ptdf_offset][row_idx[valid]].to(device)
    data.edge_attr = torch.cat([data.edge_attr, ptdf_rows], dim=-1)
    return data


def model_results(network, print_results=False, plot_results=False, compute_ptdf=False):
    """
    Run PyPSA power flow and store results in DataFrames.
    Print if print_results=True, plot if plot_results=True.

    DISTSLACK: pass distribute_slack=True to use PyPSA's distributed slack solver.
    """
    # DISTSLACK: use distribute_slack=True to properly handle distributed slack
    network.pf(use_seed=True, distribute_slack=False)  # DISTSLACK: change to True when using distributed slack

    snapshots = network.snapshots
    buses = network.buses.index
    lines = network.lines.index

    bus_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product([buses, ["P pu", "Q pu", "V pu", "Angle deg"]]),
        dtype=float,
    )
    line_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product([lines, ["P0 pu", "P1 pu", "Q0 pu", "Q1 pu"]]),
        dtype=float,
    )

    for t in snapshots:
        for bus in buses:
            bus_results.loc[t, (bus, "P pu")]    = network.buses_t.p.loc[t, bus]
            bus_results.loc[t, (bus, "Q pu")]  = network.buses_t.q.loc[t, bus]
            bus_results.loc[t, (bus, "V pu")]    = network.buses_t.v_mag_pu.loc[t, bus]
            bus_results.loc[t, (bus, "Angle deg")] = np.degrees(network.buses_t.v_ang.loc[t, bus])
        for line in lines:
            line_results.loc[t, (line, "P0 pu")]   = network.lines_t.p0.loc[t, line]
            line_results.loc[t, (line, "P1 pu")]   = network.lines_t.p1.loc[t, line]
            line_results.loc[t, (line, "Q0 pu")] = network.lines_t.q0.loc[t, line]
            line_results.loc[t, (line, "Q1 pu")] = network.lines_t.q1.loc[t, line]

    if print_results:
        print(bus_results)
        print(line_results)
    if plot_results:
        plot_network_results(network, bus_results)

    ptdf_df = None
    if compute_ptdf:
        ptdf_df = compute_ptdf_matrix(network)

    return bus_results, line_results, ptdf_df


def batched_gnn_forward(model, data_list):
    """Run GNN on a batch of graphs in a single forward pass."""
    exclude_keys = ['y_ptdf', 'ptdf_line_index', 'y_line_p']
    batch = Batch.from_data_list(data_list, exclude_keys=exclude_keys)
    out = model(batch)
    if isinstance(out, tuple):
        out = out[0]
    results = []
    for g in range(len(data_list)):
        mask = (batch.batch == g)
        results.append(out[mask])
    return results


def predict_network_results_with_masks(
    model, network, use_edge_features=True, debug=False,
    # STSI 26.04.07: learned_edge PTDF support
    ptdf_params=None, max_buses=0, ptdf_offset=0,
):
    """
    Run GNN prediction on a network and return bus/line results with prediction masks.

    DISTSLACK: prediction masks now also flag distributed slack buses correctly.
    Distributed slack buses predict P and Q (same as single slack).
    STSI 26.04.07: Augments edge_attr when ptdf_params is provided (learned_edge).

    Args:
        model:   Trained PowerFlowGNN model
        network: PyPSA network object (must have solved power flow for feature extraction)
    Returns:
        bus_results, line_results, prediction_masks
    """
    dataset = PowerFlowDataset([network], use_edge_features=use_edge_features)
    snapshots = network.snapshots
    buses = list(network.buses.index)
    bus_to_idx = {b: i for i, b in enumerate(buses)}

    bus_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product([buses, ["P pu", "Q pu", "V pu", "Angle deg"]]),
        dtype=float,
    )
    line_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product(
            [network.lines.index, ["P0 pu", "P1 pu", "Q0 pu", "Q1 pu"]]
        ),
        dtype=float,
    )
    # Prediction masks: which properties were predicted (not taken from input)
    prediction_masks = {
        "P":     pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
        "Q":     pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
        "V":     pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
        "Angle": pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
    }

    model.eval()
    with torch.no_grad():
        for t_idx, t in enumerate(snapshots):
            data = dataset[t_idx]

            # STSI 26.04.07: augment edge_attr for learned_edge models
            _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx=0, ptdf_offset=ptdf_offset)

            # Fallback: zero-pad if model expects wider edge_attr (old runs without stored ptdf_params)
            if hasattr(model, 'convs') and model.convs and hasattr(model.convs[0], 'lin_edge') and model.convs[0].lin_edge is not None:
                _expected = model.convs[0].lin_edge.in_channels
                if data.edge_attr.size(1) < _expected:
                    data.edge_attr = torch.cat([data.edge_attr, torch.zeros(data.edge_attr.size(0), _expected - data.edge_attr.size(1))], dim=1)

            node_pred, _ = model(data)  # [num_buses, 4]: vmag, vang, p, q

            pred_vmag = node_pred[:, 0]
            pred_vang = node_pred[:, 1]
            pred_p    = node_pred[:, 2]
            pred_q    = node_pred[:, 3]

            for i, bus in enumerate(buses):
                bus_type = "PQ"  # default
                connected_gens = network.generators[network.generators.bus == bus]
                if len(connected_gens) > 0:
                    ctrl = connected_gens.iloc[0]["control"]
                    if ctrl in ("Slack", "PV"):
                        bus_type = ctrl

                if bus_type == "PQ":
                    # For PQ buses: predict vmag, vang
                    pred_vmag_i = pred_vmag[i]
                    pred_vang_i = pred_vang[i]
                    pred_p_i    = network.buses_t.p.loc[t, bus]   # Known
                    pred_q_i    = network.buses_t.q.loc[t, bus]   # Known
                    prediction_masks["P"].loc[t, bus]     = False
                    prediction_masks["Q"].loc[t, bus]     = False
                    prediction_masks["V"].loc[t, bus]     = True
                    prediction_masks["Angle"].loc[t, bus] = True

                elif bus_type == "PV":
                    # For PV buses: known vmag, predict vang, q
                    pred_vmag_i = network.buses_t.v_mag_pu.loc[t, bus]  # Known
                    pred_vang_i = pred_vang[i]
                    pred_p_i    = network.buses_t.p.loc[t, bus]         # Known
                    pred_q_i    = pred_q[i]
                    prediction_masks["P"].loc[t, bus]     = False
                    prediction_masks["Q"].loc[t, bus]     = True
                    prediction_masks["V"].loc[t, bus]     = False
                    prediction_masks["Angle"].loc[t, bus] = True

                else:
                    # Slack (single or distributed): known vmag, vang; predict p, q
                    # DISTSLACK: for distributed slack, vmag is known but angle
                    # reference is only soft-fixed; we still use the known angle
                    # from the PF solution as input and predict P, Q
                    pred_vmag_i = network.buses_t.v_mag_pu.loc[t, bus]  # Known
                    pred_vang_i = network.buses_t.v_ang.loc[t, bus]     # Known (reference)
                    pred_p_i    = pred_p[i]
                    pred_q_i    = pred_q[i]
                    prediction_masks["P"].loc[t, bus]     = True
                    prediction_masks["Q"].loc[t, bus]     = True
                    prediction_masks["V"].loc[t, bus]     = False
                    prediction_masks["Angle"].loc[t, bus] = False

                bus_results.loc[t, (bus, "P pu")]      = pred_p_i.item() if torch.is_tensor(pred_p_i) else pred_p_i
                bus_results.loc[t, (bus, "Q pu")]      = pred_q_i.item() if torch.is_tensor(pred_q_i) else pred_q_i
                bus_results.loc[t, (bus, "V pu")]      = pred_vmag_i.item() if torch.is_tensor(pred_vmag_i) else pred_vmag_i
                bus_results.loc[t, (bus, "Angle deg")] = np.degrees(
                    pred_vang_i.item() if torch.is_tensor(pred_vang_i) else pred_vang_i
                )

            if debug:
                print(f"Predicted angles should be small radians: {pred_vang[:3].numpy()}")
                print(f"Predicted angles in degrees: {np.degrees(pred_vang[:3].numpy())}")
                print(f"Angle range: {pred_vang.min():.6f} to {pred_vang.max():.6f} radians")

            # Calculate line flows from predicted voltages
            vmag_np = np.array([
                bus_results.loc[t, (bus, "V pu")] for bus in buses
            ], dtype=float)
            vang_np = np.radians(np.array([
                bus_results.loc[t, (bus, "Angle deg")] for bus in buses
            ], dtype=float))
            pred_line_flows = calculate_line_flows(network, vmag_np, vang_np, t_idx)

            for i, line in enumerate(network.lines.index):
                line_results.loc[t, (line, "P0 pu")]   = pred_line_flows["p0"][i]
                line_results.loc[t, (line, "P1 pu")]   = pred_line_flows["p1"][i]
                line_results.loc[t, (line, "Q0 pu")] = pred_line_flows["q0"][i]
                line_results.loc[t, (line, "Q1 pu")] = pred_line_flows["q1"][i]

    return bus_results, line_results, prediction_masks



## Hyperparameter Sweep Helpers

In [ ]:
def evaluate_gnn_on_test_set(
    model,
    test_networks: list,
    use_edge_features: bool = True,
    use_pnet_balance: bool = True,
    device=None,
    # STSI 26.04.07: learned_edge PTDF support
    ptdf_params=None,
    max_buses: int = 0,
    ptdf_offset: int = 0,
    # [STSI260502]: batch_size for batched timing measurement
    batch_size: int = 64,
) -> dict:
    """
    Evaluate GNN on test networks snapshot-by-snapshot.

    KEY FIX (2026-03-25): After inference, override known variables with ground
    truth inputs — otherwise scatter plots show model predictions for variables
    the model was never asked to predict (Vmag/Vang at slack/PV), making results
    appear much worse than they are.

    Override rules (mirror of _masked_mse_loss):
        PV    → vmag_eval[pv]    = x[pv,    5]  (known Vmag)
        Slack → vmag_eval[slack] = x[slack,  5]  (known Vmag = v_set)
        Slack → vang_eval[slack] = x[slack,  6]  (known Vang = 0)

    STSI 26.04.07: When ptdf_params is not None (learned_edge mode), augments
    data.edge_attr with learned PTDF rows before model forward pass so the
    model receives the same edge feature width it was trained with.

    [STSI260502]: Added multi-level timing:
      solve_time_ms:       pure forward pass per snapshot
      postproc_time_ms:    mask overrides + cpu transfer + line flow calc
      total_time_ms:       solve + postproc (single-snapshot end-to-end)
      batch_solve_time_ms: batched forward pass amortized per snapshot
    Dataset construction and data.to(device) excluded from all timers.

    Returns dict with MAE/RMSE for Vmag, Vang, P, Q + line flow MAE + timing.
    """
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    dataset = PowerFlowDataset(
        test_networks,
        use_edge_features=use_edge_features,
        use_pnet_balance=use_pnet_balance,
    )

    vmag_errors, vang_errors, p_errors, q_errors = [], [], [], []
    vmag_true_all, vang_true_all, p_true_all, q_true_all = [], [], [], []

    lflow_errors = []
    lflow_q0_errors = []
    all_true_flows, all_pred_flows = [], []
    all_true_q0_flows, all_pred_q0_flows = [], []
    n_snapshots = 0

    # [STSI260502]: per-snapshot timing lists (solve and postproc measured separately)
    solve_times = []
    postproc_times = []
    # [STSI260502]: collect prepared data for batched timing later
    all_data_for_batch = []

    with torch.no_grad():
        for i in range(len(dataset)):
            data = dataset[i].to(device)

            # STSI 26.04.07: augment edge_attr for learned_edge models
            net_idx, _ = dataset._index[i]
            _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx, ptdf_offset, device)

            # [STSI260502]: save a copy for batched timing (before forward modifies anything)
            all_data_for_batch.append(data.clone())

            # [STSI260502]: Level 1 — time pure forward pass only
            t_solve_start = time.perf_counter()
            node_pred, *_ = model(data, return_embeddings=True)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t_solve = time.perf_counter() - t_solve_start

            # [STSI260502]: Level 2 — time post-processing (masks + line flows)
            t_post_start = time.perf_counter()

            # ── Extract masks ─────────────────────────────────────────────────
            slack_mask = data.slack_mask.bool()
            pv_mask    = data.pv_mask.bool()
            pq_mask    = data.pq_mask.bool()

            # ── Raw predictions ───────────────────────────────────────────────
            vmag_eval = node_pred[:, 0].clone()
            vang_eval = node_pred[:, 1].clone()
            p_eval    = node_pred[:, 2].clone()
            q_eval    = node_pred[:, 3].clone()

            # ── Override known variables with ground-truth inputs ─────────────
            # PV: Vmag is known (set by voltage regulator)
            if pv_mask.any():
                vmag_eval[pv_mask] = data.x[pv_mask, 5]

            # Slack: both Vmag and Vang=0 are known
            if slack_mask.any():
                vmag_eval[slack_mask] = data.x[slack_mask, 5]
                vang_eval[slack_mask] = data.x[slack_mask, 6]  # should be ~0

            # PQ: P and Q are known inputs
            if pq_mask.any():
                p_eval[pq_mask] = data.x[pq_mask, 3]
                q_eval[pq_mask] = data.x[pq_mask, 4]

            # PV: P is known input
            if pv_mask.any():
                p_eval[pv_mask] = data.x[pv_mask, 3]

            # ── Ground truth ──────────────────────────────────────────────────
            vmag_true = data.y[:, 0]
            vang_true = data.y[:, 1]
            p_true    = data.y[:, 2]
            q_true    = data.y[:, 3]

            vmag_errors.append((vmag_eval - vmag_true).abs().mean().item())
            vang_errors.append((vang_eval - vang_true).abs().mean().item())
            p_errors.append((p_eval - p_true).abs().mean().item())
            q_errors.append((q_eval - q_true).abs().mean().item())
            vmag_true_all.append(vmag_true.detach())
            vang_true_all.append(vang_true.detach())
            p_true_all.append(p_true.detach())
            q_true_all.append(q_true.detach())
            n_snapshots += 1

            # ── Step 4 (partial): line flow MAE ──────────────────────────────
            if test_networks is not None and hasattr(dataset, '_index') and i < len(dataset._index):
                try:
                    net_idx, t_idx = dataset._index[i]
                    network  = test_networks[net_idx]
                    snapshot = network.snapshots[t_idx]
                    if not network.lines_t.p0.empty:
                        true_flows = network.lines_t.p0.loc[snapshot].values
                        vmag_np = vmag_eval.cpu().numpy()
                        vang_np = vang_eval.cpu().numpy()
                        flows = calculate_line_flows(network, vmag_np, vang_np)
                        pred_flows = np.array(flows["p0"])  # already in p.u.
                        err = np.abs(pred_flows - true_flows).mean()
                        lflow_errors.append(err)
                        all_true_flows.append(true_flows)
                        all_pred_flows.append(pred_flows)
                    if not network.lines_t.q0.empty:
                        true_q0 = network.lines_t.q0.loc[snapshot].values
                        pred_q0 = np.array(flows["q0"])  # already in p.u.
                        lflow_q0_errors.append(np.abs(pred_q0 - true_q0).mean())
                        all_true_q0_flows.append(true_q0)
                        all_pred_q0_flows.append(pred_q0)
                except Exception as _lfe:
                    if not lflow_errors:  # only warn once
                        print(f"[line_flow_mae] skipped: {_lfe}")

            # [STSI260502]: end post-processing timer (includes masks + line flows)
            t_post = time.perf_counter() - t_post_start
            solve_times.append(t_solve)
            postproc_times.append(t_post)

    # [STSI260502]: Phase 2 — batched timing measurement
    # Warmup batched forward
    n_batch_warmup = 3
    with torch.no_grad():
        if all_data_for_batch:
            for _ in range(n_batch_warmup):
                _ = batched_gnn_forward(model, all_data_for_batch[:batch_size])
            
            batch_times = []
            batch_counts = []
            for start in range(0, len(all_data_for_batch), batch_size):
                chunk = all_data_for_batch[start:start + batch_size]
                t0 = time.perf_counter()
                _ = batched_gnn_forward(model, chunk)
                if device.type == 'cuda':
                    torch.cuda.synchronize()
                batch_times.append(time.perf_counter() - t0)
                batch_counts.append(len(chunk))
            batch_ms_per_snap = sum(batch_times) * 1000 / max(1, sum(batch_counts))
        else:
            batch_ms_per_snap = float('nan')

    metrics = {
        "vmag_mae":      float(np.mean(vmag_errors)),
        "vang_mae":      float(np.mean(vang_errors)),
        "p_mae":         float(np.mean(p_errors)),
        "q_mae":         float(np.mean(q_errors)),
        "vmag_rmse":     float(np.sqrt(np.mean([e**2 for e in vmag_errors]))),
        "vang_rmse":     float(np.sqrt(np.mean([e**2 for e in vang_errors]))),
        "vmag_true_std": float(torch.cat(vmag_true_all).std().item()),
        "vang_true_std": float(torch.cat(vang_true_all).std().item()),
        "p_true_std":    float(torch.cat(p_true_all).std().item()),
        "q_true_std":    float(torch.cat(q_true_all).std().item()),
    }
    if lflow_errors:
        metrics["line_flow_mae"] = float(np.mean(lflow_errors))
    if lflow_q0_errors:
        metrics["line_flow_q0_mae"] = float(np.mean(lflow_q0_errors))

    if all_true_flows:
        raw = {
            "p0_true": np.concatenate(all_true_flows),
            "p0_pred": np.concatenate(all_pred_flows),
        }
        if all_true_q0_flows:
            raw["q0_true"] = np.concatenate(all_true_q0_flows)
            raw["q0_pred"] = np.concatenate(all_pred_q0_flows)
        metrics["line_flow_raw"] = raw
        
    print("----------------\n Evaluation Metrics (known variables overridden) ---------------")
    for k, v in metrics.items():
        if isinstance(v, (int, float)):
            print(f"  {k:20s}: {v:.6f}")

    # [STSI260502]: monolithic timing kept for backward compat
    inference_time_s = sum(solve_times) + sum(postproc_times)
    metrics["inference_time_s"] = inference_time_s
    metrics["inference_time_per_snapshot_ms"] = (
        1000.0 * inference_time_s / max(1, n_snapshots)
    )
    
    # [STSI260502]: new multi-level timing metrics
    solve_ms    = float(np.mean(solve_times)) * 1000
    postproc_ms = float(np.mean(postproc_times)) * 1000
    metrics["solve_time_ms"]       = solve_ms
    metrics["postproc_time_ms"]    = postproc_ms
    metrics["total_time_ms"]       = solve_ms + postproc_ms
    metrics["batch_solve_time_ms"] = batch_ms_per_snap

    print(f"\n  --- Timing breakdown (per snapshot) ---")
    print(f"  {'solve_time_ms':20s}: {solve_ms:.3f}")
    print(f"  {'postproc_time_ms':20s}: {postproc_ms:.3f}")
    print(f"  {'total_time_ms':20s}: {solve_ms + postproc_ms:.3f}")
    print(f"  {'batch_solve_time_ms':20s}: {batch_ms_per_snap:.3f}")

    return metrics




def load_sweep_results(pkl_path: str) -> list:
    """
    Load a full sweep result list from a pickle file.
    Returns the runs list as saved by save_sweep_results or run_hparam_sweep.
    """
    with open(pkl_path, "rb") as f:
        runs = pickle.load(f)
    print(f"Loaded {len(runs)} runs from {pkl_path}")
    return runs




In [ ]:
# =============================================================================
# SECTION 11: HYPERPARAMETER SWEEP
# =============================================================================

def run_hparam_sweep(
    networks,
    physics_configs: List[PhysicsConfig] = None,
    batch_sizes=(32,),
    learning_rates=(1e-3,),
    num_epochs=50,
    hidden_dims=(64,),
    num_layers_list=(3,),
    conv_types=("gatv2",),
    head_modes=("standard",), # or "with encoder"
    seeds=(42,),
    warmup_epochs_list=(10,),
    y_matrix_sources=("manual",),
    tag="run",
    weight_ptdf=0.0,        # added for v2.1.2 compatibility; now should be set via PhysicsConfig
    ptdf_loss_mode="matrix",# added for v2.1.2 compatibility; now should be set via PhysicsConfig
    ptdf_alpha=0.5,# added for v2.1.2 compatibility; now should be set via PhysicsConfig
    verbose=False,
    save_path=None,
):
    """
    Run a hyperparameter sweep over PhysicsConfig and model/training settings.

    Resume / checkpoint behavior
    ----------------------------
    If save_path is provided and exists, previously completed runs are loaded and
    skipped using a stable run_key. After each successful run, the full runs list
    is checkpointed back to save_path.

    Also stores training_time per run and uses previously completed runtimes
    (including resumed runs) to estimate remaining time and finish clock time.
    """
    import itertools
    import os
    import pickle
    import sys
    import time
    from datetime import datetime, timedelta

    if physics_configs is None:
        physics_configs = [PhysicsConfig(w_phys=0.0)]

    combos = list(itertools.product(
        physics_configs,
        batch_sizes,
        learning_rates,
        hidden_dims,
        num_layers_list,
        conv_types,
        y_matrix_sources,
        warmup_epochs_list,
        seeds,
        head_modes
    ))

    total_runs = len(combos)
    runs = []
    done_keys = set()
    runtimes = []

    sweep_start_time = time.time()

    # ------------------------------------------------------------------
    # Resume from checkpoint if available
    # ------------------------------------------------------------------
    if save_path and os.path.isfile(save_path):
        with open(save_path, "rb") as f:
            runs = pickle.load(f)

        backfilled = 0
        restored_runtimes = 0

        for r in runs:
            if not r.get("run_key"):
                physics_label = r.get("physics_label", "baseline")
                bs = r.get("batch_size")
                lr = r.get("lr")
                hidden_dim = r.get("hidden_dim")
                num_layers = r.get("num_layers")
                conv_type = r.get("conv_type")
                ysrc = r.get("y_matrix_source")
                warmup_epochs = r.get("warmup_epochs", 10)
                seed = r.get("seed", 42)
                head_modes = r.get("head_mode", "standard")

                r["run_key"] = (
                    f"{physics_label}_"
                    f"bs{bs}_"
                    f"lr{lr:.2e}_"
                    f"h{hidden_dim}_"
                    f"L{num_layers}_"
                    f"{conv_type}_"
                    f"{ysrc}_"
                    f"wu{warmup_epochs}_"
                    f"s{seed}"
                    f"_head{head_modes}"
                )
                backfilled += 1

            done_keys.add(r["run_key"])

            t = r.get("training_time", None)
            if isinstance(t, (int, float)) and t > 0:
                runtimes.append(float(t))
                restored_runtimes += 1

        print(
            f"[checkpoint] Loaded {len(runs)} completed runs; "
            f"{len(done_keys)} keys will be skipped."
        )
        if backfilled:
            print(f"[checkpoint] Added run_key to {backfilled} legacy runs.")
        if restored_runtimes:
            avg_loaded = sum(runtimes) / len(runtimes)
            print(
                f"[checkpoint] Restored {restored_runtimes} prior training_time values "
                f"(avg {avg_loaded/60:.1f} min/run)."
            )

    def fmt_seconds(seconds):
        if seconds is None:
            return "calculating..."
        seconds = int(max(0, seconds))
        m, s = divmod(seconds, 60)
        h, m = divmod(m, 60)
        if h > 0:
            return f"{h}h {m}m {s}s"
        elif m > 0:
            return f"{m}m {s}s"
        else:
            return f"{s}s"

    def fmt_clock_time(unix_ts):
        return datetime.fromtimestamp(unix_ts).strftime("%Y-%m-%d %H:%M:%S")

    for current_run, (
        pcfg,
        bs,
        lr,
        hidden_dim,
        num_layers,
        conv_type,
        ysrc,
        warmup_epochs,
        seed,
        head_mode,
    ) in enumerate(combos, 1):

        key = make_run_key(
            pcfg=pcfg,
            bs=bs,
            lr=lr,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            conv_type=conv_type,
            ysrc=ysrc,
            warmup_epochs=warmup_epochs,
            seed=seed,
            head_mode=head_mode,
        )

        if key in done_keys:
            print(f"[skip {current_run}/{total_runs}] already done: {key}")
            continue

        run_start = time.time()

        if runtimes:
            avg_time = sum(runtimes) / len(runtimes)
            median_time = sorted(runtimes)[len(runtimes) // 2]
            remaining_runs_including_this = total_runs - current_run + 1
            eta_seconds = median_time * remaining_runs_including_this
            estimated_total_seconds = median_time * total_runs
            estimated_finish_ts = time.time() + eta_seconds
        else:
            avg_time = None
            median_time = None
            eta_seconds = None
            estimated_total_seconds = None
            estimated_finish_ts = None

        mode = getattr(pcfg, "physics_mode", "rich")
        parts = [
            f"{current_run}/{total_runs}",
            f"mode={mode}",
            f"cfg={pcfg.label()}",
            f"wphys={pcfg.w_phys}",
            f"ptdfw={pcfg.weight_ptdf}",
            f"ptdfmode={pcfg.ptdf_loss_mode}",
            f"ptdfbr={pcfg.ptdf_branch_mode}",
            f"bs={bs}",
            f"lr={lr:.0e}",
            f"warmup={warmup_epochs}",
            f"seed={seed}",
        ]
        parts.append(f"Starting time={fmt_clock_time(time.time())}")
        if eta_seconds is not None:
            parts.append(f"remaining={fmt_seconds(eta_seconds)}")
            parts.append(f"est_total={fmt_seconds(estimated_total_seconds)}")
            parts.append(f"finish≈{fmt_clock_time(estimated_finish_ts)}")
        else:
            parts.append("remaining=calculating...")
            parts.append("finish≈calculating...")

        print(" | ".join(parts))

        if not verbose:
            old_stdout, old_stderr = sys.stdout, sys.stderr
            sys.stdout = open(os.devnull, "w")
            sys.stderr = open(os.devnull, "w")

        try:
            model, history, test_metrics, debug_objects = train_power_flow_gnn(
                networks=networks,
                num_epochs=num_epochs,
                batch_size=bs,
                lr=lr,
                physics_cfg=pcfg,
                conv_type=conv_type,
                hidden_dim=hidden_dim,
                num_layers=num_layers,
                use_edge_features=True,
                seed=seed,
                warmup_epochs=warmup_epochs,
                y_matrix_source=ysrc,
                weight_ptdf=weight_ptdf,          # ← scalar from your grid added for testing only
                ptdf_loss_mode=ptdf_loss_mode,    # ← e.g. "matrix"
                ptdf_alpha=ptdf_alpha,            # ← used only for "mixed"
                tqdm_file=old_stderr if not verbose else None,
                return_debug_objects=True,
                head_mode=head_mode, # [STSI060526] Added to toggle between standard and multi-head output modes for node types
            )
        finally:
            if not verbose:
                sys.stdout.close()
                sys.stderr.close()
                sys.stdout, sys.stderr = old_stdout, old_stderr

        runtime = time.time() - run_start
        runtimes.append(runtime)

        final_train_loss = (
            history["train_total"][-1]
            if "train_total" in history and len(history["train_total"]) > 0
            else float("nan")
        )
        final_val_loss = (
            history["val_total"][-1]
            if "val_total" in history and len(history["val_total"]) > 0
            else float("nan")
        )

        run_info = {
            "tag": tag,

            # physics / PTDF config
            "physics_mode": pcfg.physics_mode,
            "physics_label": pcfg.label(),
            "weight_physics": pcfg.w_phys,
            "use_power_balance": pcfg.use_power_balance,
            "use_angle_ref_penalty": pcfg.use_angle_ref_penalty,
            "use_q_partial_mode": pcfg.use_q_partial_mode,
            "use_ptdf_loss_flag": pcfg.use_ptdf_loss,
            "weight_ptdf": pcfg.weight_ptdf,
            "ptdf_loss_mode": pcfg.ptdf_loss_mode,
            "ptdf_alpha": pcfg.ptdf_alpha,
            "ptdf_branch_mode": pcfg.ptdf_branch_mode,

            # run hyperparameters
            "batch_size": bs,
            "lr": lr,
            "conv_type": conv_type,
            "hidden_dim": hidden_dim,
            "num_layers": num_layers,
            "warmup_epochs": warmup_epochs,
            "seed": seed,
            "y_matrix_source": ysrc,
            "head_mode": head_mode, # [STSI060526] Added to toggle between standard and multi-head output modes for node types

            # outputs
            "history": history,
            "test_metrics": test_metrics,
            "model": model,
            "training_time": runtime,
            "final_train_loss": final_train_loss,
            "final_val_loss": final_val_loss,

            # STSI 26.04.07: learned_edge artefacts for inference
            "_ptdf_params": test_metrics.get("_ptdf_params"),
            "_max_buses":   test_metrics.get("_max_buses", 0),

            # resume / dedup
            "run_key": key,

            # test split for later evaluation
            "test_networks": debug_objects.get("test_networks"),
            "num_scenarios":   len(networks),
            "bus_count_min":   min(len(n.buses) for n in networks),
            "bus_count_max":   max(len(n.buses) for n in networks),
            "bus_count_range": max(len(n.buses) for n in networks) - min(len(n.buses) for n in networks),
        }

        runs.append(run_info)
        done_keys.add(key)

        tm = test_metrics
        print(
            f"→ done in {runtime/60:.1f} min | "
            f"Train {final_train_loss:.4f} "
            f"Val {final_val_loss:.4f} "
            f"Test {tm.get('test_total', float('nan')):.4f} "
        )

        # ------------------------------------------------------------------
        # Checkpoint save after every completed run
        # ------------------------------------------------------------------
        if save_path:
            # Strip unpicklable objects (PyPSA networks contain weakrefs)
            _stashed = []
            for r in runs:
                _stashed.append(r.pop("test_networks", None))
            # Atomic write: dump to .tmp first, then rename so an interrupt
            # never leaves the real checkpoint file truncated/empty
            _tmp_path = save_path + ".tmp"
            with open(_tmp_path, "wb") as f:
                pickle.dump(runs, f)
            os.replace(_tmp_path, save_path)
            for r, tn in zip(runs, _stashed):
                if tn is not None:
                    r["test_networks"] = tn

    return runs

def make_run_key(
    pcfg,
    bs,
    lr,
    hidden_dim,
    num_layers,
    conv_type,
    ysrc,
    warmup_epochs,
    seed,
    head_mode: str = "standard",
) -> str:
    """
    Stable string key identifying one combo used for checkpoint deduplication
    in run_hparam_sweep.
    """
    return (
        f"{pcfg.label()}_"
        f"bs{bs}_"
        f"lr{lr:.2e}_"
        f"h{hidden_dim}_"
        f"L{num_layers}_"
        f"{conv_type}_"
        f"{ysrc}_"
        f"wu{warmup_epochs}_"
        f"s{seed}_"
        f"hm{head_mode}"
    )

def create_comparison_dataframe(runs_list):
    """Create a comprehensive comparison DataFrame from sweep results."""
    rows = []
    for r in runs_list:
        tm = r["test_metrics"]
        rows.append({
            "tag":           r["tag"],
            "w_phys":        r["weight_physics"],
            "w_ptdf":        r["weight_ptdf"],
            "bs":            r["batch_size"],
            "lr":            r["lr"],
            "train_loss":    r["final_train_loss"],
            "val_loss":      r["final_val_loss"],
            "test_total":    tm["test_total"],
            "test_mse":      tm["test_mse"],
            "test_physics":  tm["test_physics"],
            "test_ptdf":     tm["test_ptdf"],
            "training_time": r["training_time"],
        })
    return pd.DataFrame(rows)

def _get_run_styles(runs):
    varying = [
        k for k in ("physics_mode", "physics_label", "weight_ptdf",
                    "batch_size", "lr", "conv_type", "y_matrix_source")
        if len({str(x.get(k)) for x in runs}) > 1
    ]
    if not varying:
        varying = ["physics_label"]

    dim_values = {}
    for k in varying:
        seen = {}
        for r in runs:
            v = str(r.get(k, ""))
            if v not in seen:
                seen[v] = len(seen)
        dim_values[k] = seen

    styles = {}
    for r in runs:
        rid = id(r)
        c_idx  = dim_values[varying[0]][str(r.get(varying[0], ""))] if len(varying) >= 1 else 0
        ls_idx = dim_values[varying[1]][str(r.get(varying[1], ""))] if len(varying) >= 2 else 0
        mk_idx = dim_values[varying[2]][str(r.get(varying[2], ""))] if len(varying) >= 3 else 0

        styles[rid] = (
            _COLOR_CYCLE[c_idx  % len(_COLOR_CYCLE)],
            _LINE_STYLES[ls_idx % len(_LINE_STYLES)],
            _MARKERS[mk_idx     % len(_MARKERS)],
        )
    return styles

def build_legend_label(r, runs):
    varying = [
        k for k in [
            "physics_mode",
            "physics_label",
            "weight_ptdf",
            "ptdf_loss_mode",
            "ptdf_branch_mode",
            "ptdf_alpha",
            "batch_size",
            "lr",
            "conv_type",
            "y_matrix_source",
            "warmup_epochs",
            "hidden_dim",
            "num_layers",
        ]
        if len({str(x.get(k)) for x in runs}) > 1
    ]

    if not varying:
        varying = ["physics_label"]

    parts = []

    if "physics_mode" in varying:
        parts.append(r.get("physics_mode", "rich"))
    if "physics_label" in varying:
        parts.append(r.get("physics_label", ""))

    if "weight_ptdf" in varying:
        parts.append(f"ptdf={r.get('weight_ptdf')}")
    if "ptdf_loss_mode" in varying:
        parts.append(f"mode={r.get('ptdf_loss_mode')}")
    if "ptdf_branch_mode" in varying:
        parts.append(f"br={r.get('ptdf_branch_mode')}")
    if "ptdf_alpha" in varying and r.get("ptdf_loss_mode") == "mixed":
        parts.append(f"a={r.get('ptdf_alpha')}")

    if "y_matrix_source" in varying:
        parts.append(f"y={r.get('y_matrix_source')}")
    if "warmup_epochs" in varying:
        parts.append(f"wu={r.get('warmup_epochs')}")
    if "hidden_dim" in varying:
        parts.append(f"h={r.get('hidden_dim')}")
    if "num_layers" in varying:
        parts.append(f"L={r.get('num_layers')}")
    if "batch_size" in varying:
        parts.append(f"bs={r.get('batch_size')}")
    if "lr" in varying:
        parts.append(f"lr={r.get('lr'):.0e}")
    if "conv_type" in varying:
        parts.append(r.get("conv_type", ""))

    return " | ".join(parts)


def plot_all_runs_training_curves(runs, title_prefix="All runs", marker_every=10):
    """
    Plot train/val loss for all runs.
    - Color    separates the first varying hyperparameter (e.g. physics_label).
    - Linestyle separates the second varying hyperparameter (e.g. weight_ptdf).
    - Marker   separates the third varying hyperparameter (e.g. batch_size / lr).
    - marker_every: place a marker every N epochs (keeps plots readable).
    """
    styles = _get_run_styles(runs)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 10))

    for r in runs:
        label = build_legend_label(r, runs)
        h     = r["history"]
        color, ls, mk = styles[id(r)]
        n = len(h["train_total"])
        markevery = max(1, n // marker_every) if mk else None

        ax1.plot(h["train_total"], label=label, alpha=0.85,
                 color=color, linestyle=ls, marker=mk,
                 markevery=markevery, markersize=5)
        ax2.plot(h["val_total"],   label=label, alpha=0.85,
                 color=color, linestyle=ls, marker=mk,
                 markevery=markevery, markersize=5)

    for ax, title, ylabel in [
        (ax1, f"{title_prefix} — Training Loss",   "Training Loss"),
        (ax2, f"{title_prefix} — Validation Loss", "Validation Loss"),
    ]:
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.legend(fontsize=8, ncol=max(1, len(runs) // 8))
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def plot_loss_components(runs, title_prefix="Loss Components", marker_every=10):
    """
    Plot train/val loss for each component (MSE, Physics, Angle-ref, PTDF)
    in a 2×2 grid of subplots, with each subplot showing train (solid) and
    val (dashed) curves for all runs.
    """
    styles = _get_run_styles(runs)

    components = [
        ("mse",       "MSE Loss"),
        ("phys",      "Physics Loss"),
        ("angle_ref", "Angle-Ref Loss"),
        ("ptdf",      "PTDF Loss"),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(24, 14))
    axes = axes.flatten()

    for ax, (key, comp_title) in zip(axes, components):
        for r in runs:
            label = build_legend_label(r, runs)
            h = r["history"]
            train_key = f"train_{key}"
            val_key   = f"val_{key}"

            if train_key not in h or not h[train_key]:
                continue

            color, ls, mk = styles[id(r)]
            n = len(h[train_key])
            markevery = max(1, n // marker_every) if mk else None

            ax.plot(h[train_key], label=f"{label} (train)", alpha=0.85,
                    color=color, linestyle="-", marker=mk,
                    markevery=markevery, markersize=5)
            ax.plot(h[val_key],   label=f"{label} (val)", alpha=0.85,
                    color=color, linestyle="--", marker=mk,
                    markevery=markevery, markersize=5)

        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title(f"{title_prefix} — {comp_title}")
        ax.legend(fontsize=7, ncol=max(1, len(runs) // 4))
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def plot_all_runs_accuracy_metrics(runs, title_prefix="All runs"):
    """
    Bar chart comparison of test accuracy metrics across runs.
    Reads from r["test_metrics"]["overall_metrics"] for mean/std per metric.
    Metrics: vmag_mae, vang_rmse, p_mae, q_mae.
    """
    metrics = [
        ("vmag_mae",  "Vmag MAE  [p.u.]"),
        ("vang_rmse", "Vang RMSE [rad]"),
        ("p_mae",     "P MAE     [p.u.]"),
        ("q_mae",     "Q MAE     [p.u.]"),
        ("line_flow_mae", "Line Flow MAE [p.u.]"),
        ("line_flow_q0_mae", "Line Flow Q0 MAE [p.u.]"),
    ]
    x      = list(range(len(runs)))
    labels = [build_legend_label(r, runs) for r in runs]
    styles = _get_run_styles(runs)   # reuse the same color assignment as training curves
    colors = [styles[id(r)][0] for r in runs]

    fig, axes = plt.subplots(1, len(metrics), figsize=(max(16, 2.5 * len(metrics) + len(runs)), 6))
    fig.suptitle(f"{title_prefix} — Test Accuracy Metrics", fontsize=13)
    short_labels = [str(i + 1) for i in range(len(runs))]   # "1", "2", …
    for ax, (metric_key, metric_label) in zip(axes, metrics):
        means, stds = [], []
        for r in runs:
            tm = r.get("test_metrics", {})
            val = tm.get(metric_key, float("nan"))
            # support both scalar and dict-with-mean
            if isinstance(val, dict):
                mean = float(val.get("mean", float("nan")))
                std  = float(val.get("std",  0.0))
            else:
                mean = float(val)
                std  = 0.0
            means.append(mean)
            stds.append(std)

        bars = ax.bar(x, means, yerr=stds, capsize=5, alpha=0.8, color=colors)
        ax.set_xticks(x)
        ax.set_xticklabels(short_labels, fontsize=9)
        ax.set_ylabel(metric_label)
        ax.set_title(metric_label)
        ax.grid(True, axis="y", alpha=0.3)

        valid_means = [m for m in means if m == m]  # filter nan
        top = max(valid_means) if valid_means else 1.0
        for bar, mean in zip(bars, means):
            if mean == mean:  # not nan
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + top * 0.02,
                    f"{mean:.4f}",
                    ha="center", va="bottom", fontsize=7
                )

    # ── Shared legend below all subplots ─────────────────────────────────────
    legend_lines = [
        plt.Line2D([0], [0], color=colors[i], linewidth=6, alpha=0.8,
                   label=f"{i+1}: {labels[i]}")
        for i in range(len(runs))
    ]
    fig.legend(
        handles=legend_lines,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.02),
        ncol=max(1, min(3, len(runs))),
        fontsize=8,
        frameon=True,
    )
    plt.tight_layout(rect=[0, 0.12 + 0.03 * (len(runs) // 3), 1, 1])
    plt.show()

def print_sweep_summary(runs_list, top_n=5):
    """
    Print a ranked summary table of hyperparameter sweep results.
    Shows top_n configurations by test loss.
    """
    sorted_runs = sorted(runs_list, key=lambda r: r["test_metrics"]["test_total"])
    print(f"{'Rank':<5}{'w_phys':<9}{'w_ptdf':<9}{'bs':<6}{'lr':<11}"
          f"{'Train':<11}{'Val':<11}{'Test':<11}{'Time(s)':<9}")
    print("-" * 92)
    for i, run in enumerate(sorted_runs[:top_n], 1):
        tm = run["test_metrics"]
        print(
            f"{i:<5}{run['weight_physics']:<9}{run['weight_ptdf']:<9}"
            f"{run['batch_size']:<6}{run['lr']:<11.2e}"
            f"{run['final_train_loss']:<11.6f}{run['final_val_loss']:<11.6f}"
            f"{tm['test_total']:<11.6f}{run['training_time']:<9.1f}"
        )

def _sanitise_filename(s: str) -> str:
    """Replace characters illegal on Windows filesystems with safe alternatives."""
    for ch in r'\/:*?"<>|=':
        s = s.replace(ch, "-")
    return s


def save_sweep_results(
    runs: list,
    save_dir: str = "training_results_saved",
    tag: str = None,
    save_models: bool = True,
    save_csv: bool = True,
    save_histories: bool = True,
):
    """
    Save all results from run_hparam_sweep to disk.
    # SECTION: save_sweep_results — updated for current run_info dict structure
    # Changes from old version:
    #   - weight_physics → w_phys (PhysicsConfig field name)
    #   - Added warmup_epochs, physics_mode, seed to CSV
    #   - Filename sanitisation (removes |, =, illegal Windows chars)
    #   - Flattened overall_metrics access matches evaluate_gnn_on_test_set output
    #   - Model label uses physics_label instead of weight_physics
    # 2026-03-26
    # =============================================================================

    Saves:
      - runs_{tag}_{ts}.pkl            : full run list (models + histories + metrics)
      - runs_{tag}_{ts}_metrics.csv    : flat metrics table
      - runs_{tag}_{ts}_histories/     : per-run training curves as CSV
      - models/runs_{tag}_{ts}/        : per-run model .pt files
      - runs_{tag}_{ts}_manifest.json  : metadata
    """
    os.makedirs(save_dir, exist_ok=True)

    if tag is None:
        tag = runs[0].get("tag", "sweep") if runs else "sweep"

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_name = _sanitise_filename(f"runs_{tag}_{timestamp}")

    # ── 1. Full pickle ────────────────────────────────────────────────────────
    pkl_path = os.path.join(save_dir, f"{base_name}.pkl")
    with open(pkl_path, "wb") as f:
        pickle.dump(runs, f)
    print(f"  [OK] Full run list  -> {pkl_path}")

    # ── 2. Metrics CSV ────────────────────────────────────────────────────────
    csv_path = None
    if save_csv:
        rows = []
        for i, r in enumerate(runs):
            tm  = r.get("test_metrics", {})
            om  = tm.get("overall_metrics", {})   # from evaluate_gnn_on_test_set

            rows.append({
                # Identifiers
                "run_idx":              i,
                "tag":                  r.get("tag", ""),
                "seed":                 r.get("seed"),
                # Physics config
                "physics_label":        r.get("physics_label"),
                "physics_mode":         r.get("physics_mode"),
                "w_phys":               r.get("weight_physics"),   # stored as weight_physics in run_info
                "use_power_balance":    r.get("use_power_balance"),
                "use_angle_ref":        r.get("use_angle_ref_penalty"),
                "use_q_partial":        r.get("use_q_partial_mode"),
                "use_ptdf_loss_flag":   r.get("use_ptdf_loss_flag"),
                # Other hparams
                "w_ptdf":               r.get("weight_ptdf"),
                "batch_size":           r.get("batch_size"),
                "lr":                   r.get("lr"),
                "conv_type":            r.get("conv_type"),
                "ptdf_loss_mode":       r.get("ptdf_loss_mode"),
                "hidden_dim":           r.get("hidden_dim"),
                "num_layers":           r.get("num_layers"),
                "warmup_epochs":        r.get("warmup_epochs"),
                "y_matrix_source":      r.get("y_matrix_source"),
                # Training outcomes
                "final_train_loss":     r.get("final_train_loss"),
                "final_val_loss":       r.get("final_val_loss"),
                "training_time_s":      r.get("training_time"),
                # Test losses
                "test_total":           tm.get("test_total"),
                "test_mse":             tm.get("test_mse"),
                "test_physics":         tm.get("test_physics"),
                "test_ptdf":            tm.get("test_ptdf"),
                # Node-wise metrics from evaluate_gnn_on_test_set
                "vmag_mae":             tm.get("vmag_mae"),
                "vang_mae":             tm.get("vang_mae"),
                "p_mae":                tm.get("p_mae"),
                "q_mae":                tm.get("q_mae"),
                "vmag_rmse":            tm.get("vmag_rmse"),
                "vang_rmse":            tm.get("vang_rmse"),
                "line_flow_mae":        tm.get("line_flow_mae"),   # Step 4 — None if not computed
                "inference_time_s":     tm.get("inference_time_s"),
                "infer_ms_per_snap":    tm.get("inference_time_per_snapshot_ms"),
            })

        csv_path = os.path.join(save_dir, f"{base_name}_metrics.csv")
        pd.DataFrame(rows).to_csv(csv_path, index=False)
        print(f"  [OK] Metrics CSV    -> {csv_path}")

    # ── 3. Per-run training histories ─────────────────────────────────────────
    if save_histories:
        hist_dir = os.path.join(save_dir, f"{base_name}_histories")
        os.makedirs(hist_dir, exist_ok=True)
        for i, r in enumerate(runs):
            h = r.get("history", {})
            if not h:
                continue
            hist_label = _sanitise_filename(
                f"run{i:03d}"
                f"_{r.get('physics_label', 'nophys')}"
                f"_pt{r.get('weight_ptdf')}"
                f"_bs{r.get('batch_size')}"
                f"_lr{r.get('lr'):.0e}"
                f"_s{r.get('seed')}"
            )
            pd.DataFrame(h).to_csv(
                os.path.join(hist_dir, f"{hist_label}.csv"),
                index_label="epoch"
            )
        print(f"  [OK] Histories      -> {hist_dir}/  ({len(runs)} files)")

    # ── 4. Model state dicts ──────────────────────────────────────────────────
    if save_models:
        model_dir = os.path.join(save_dir, "models", base_name)
        os.makedirs(model_dir, exist_ok=True)
        for i, r in enumerate(runs):
            model = r.get("model")
            if model is None:
                continue
            model_label = _sanitise_filename(
                f"run{i:03d}"
                f"_{r.get('physics_label', 'nophys')}"
                f"_pt{r.get('weight_ptdf')}"
                f"_bs{r.get('batch_size')}"
                f"_lr{r.get('lr'):.0e}"
                f"_{r.get('conv_type')}"
                f"_{r.get('ptdf_loss_mode')}"
                f"_s{r.get('seed')}"
                f".pt"
            )
            safe_torch_save(
                model.state_dict(),
                os.path.join(model_dir, model_label)
            )
        print(f"  [OK] Model weights  -> {model_dir}/  ({len(runs)} files)")

    # ── 5. Manifest ───────────────────────────────────────────────────────────
    manifest = {
        "timestamp":  timestamp,
        "tag":        tag,
        "num_runs":   len(runs),
        "save_dir":   save_dir,
        "files": {
            "pickle":    os.path.basename(pkl_path),
            "metrics":   os.path.basename(csv_path) if csv_path else None,
            "histories": f"{base_name}_histories/" if save_histories else None,
            "models":    f"models/{base_name}/" if save_models else None,
        },
        "hyperparams": {
            "physics_labels": sorted(set(r.get("physics_label", "") for r in runs)),
            "w_phys":         sorted(set(r.get("weight_physics", 0.0) for r in runs)),
            "w_ptdf":         sorted(set(r.get("weight_ptdf", 0.0) for r in runs)),
            "batch_sizes":    sorted(set(r.get("batch_size") for r in runs)),
            "learning_rates": sorted(set(r.get("lr") for r in runs)),
            "conv_types":     sorted(set(r.get("conv_type", "") for r in runs)),
            "ptdf_modes":     sorted(set(r.get("ptdf_loss_mode", "") for r in runs)),
            "warmup_epochs":  sorted(set(r.get("warmup_epochs", 0) for r in runs)),
            "seeds":          sorted(set(r.get("seed") for r in runs)),
        }
    }
    manifest_path = os.path.join(save_dir, f"{base_name}_manifest.json")
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2, default=str)
    print(f"  [OK] Manifest       -> {manifest_path}")
    print(f"\n  Saved {len(runs)} runs to '{save_dir}/{base_name}*'")



## Visualisation Functions

## Load Networks

In [16]:
networks_ieee9_test_singel_slack=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "ieee9_test.json"))

Loaded 10 networks from C:\Users\STSI\OneDrive - USN\Data_PF_GNN\training_networks_saved\ieee9_test.json


In [51]:
networks_ieee9_500_singel_slack=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "ieee9_500_wide.json"))

Loaded 500 networks from C:\Users\STSI\OneDrive - USN\Data_PF_GNN\training_networks_saved\ieee9_500_wide.json


In [13]:
networks_ieee9_mega_singel_slack=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "ieee9_mega.json"))

Loaded 1500 networks from C:\Users\STSI\OneDrive - USN\Data_PF_GNN\training_networks_saved\ieee9_mega.json


In [14]:
networks_cigre14_mega_singel_slack=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "cigre14_mega.json"))

Loaded 1500 networks from C:\Users\STSI\OneDrive - USN\Data_PF_GNN\training_networks_saved\cigre14_mega.json


In [15]:
networks_ieee30_mega_singel_slack=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "ieee30_mega.json"))

Loaded 1500 networks from C:\Users\STSI\OneDrive - USN\Data_PF_GNN\training_networks_saved\ieee30_mega.json


In [ ]:
networks_ieee30_500_wide=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "ieee30_500_wide.json"))

## Sweep Configs

In [ ]:
ptdf_comparison_configs = [
    # ── 1. Pure MSE reference (no physics, no PTDF) ──
    PhysicsConfig(
        w_phys=0.0,
        use_ptdf_loss=False,
        weight_ptdf=0.0,
    ),

    # ── 2. Best physics reference (no PTDF) ──
    PhysicsConfig(
        w_phys=0.04,
        physics_mode="rich",
        use_power_balance=True,
        use_angle_ref_penalty=True,
        use_ptdf_loss=False,
        weight_ptdf=0.0,
    ),
    # ── 3. Best physics reference (no PTDF) ──
    PhysicsConfig(
        w_phys=0.06,
        physics_mode="rich",
        use_power_balance=True,
        use_angle_ref_penalty=True,
        use_ptdf_loss=False,
        weight_ptdf=0.0,
    ),


    # ── 4. PTDF flows mode ──
    PhysicsConfig(
        w_phys=0.04,
        physics_mode="rich",
        use_power_balance=True,
        use_angle_ref_penalty=True,
        use_ptdf_loss=True,
        weight_ptdf=0.1,
        ptdf_loss_mode="flows",
    ),

        # ── 7. PTDF flows - increased weight ──
    PhysicsConfig(
        w_phys=0.04,
        physics_mode="rich",
        use_power_balance=True,
        use_angle_ref_penalty=True,
        use_ptdf_loss=True,
        weight_ptdf=0.3,
        ptdf_loss_mode="flows",
    ),
    # ── 8. PTDF flows - only (no physics) ──
    PhysicsConfig(
        w_phys=0.0,
        physics_mode="rich",
        use_power_balance=True,
        use_angle_ref_penalty=True,
        use_ptdf_loss=True,
        weight_ptdf=0.1,
        ptdf_loss_mode="flows",
    ),
]

In [45]:
mixed_configs = [
    # ── 1. Pure MSE reference (no physics, no PTDF) ──
    #PhysicsConfig(
    #    w_phys=0.0,
    #    use_ptdf_loss=False,
    #   weight_ptdf=0.0,
    #),

    # ── 2. Best physics reference (no PTDF) ──
    #PhysicsConfig(
    #    w_phys=0.04,
    #    physics_mode="rich",
    #    use_power_balance=True,
    #    use_angle_ref_penalty=True,
    #    use_ptdf_loss=False,
    #    weight_ptdf=0.0,
    #),
    # ── 3. Best physics reference (no PTDF) ──
    #PhysicsConfig(
    #   w_phys=0.06,
    #    physics_mode="rich",
    #    use_power_balance=True,
    #    use_angle_ref_penalty=True,
    #    use_ptdf_loss=False,
    #    weight_ptdf=0.0,
    #),


    # ── 4. PTDF flows mode ──
    PhysicsConfig(
        w_phys=0.04,
        physics_mode="rich",
        use_power_balance=True,
        use_angle_ref_penalty=True,
        use_ptdf_loss=True,
        weight_ptdf=0.1,
        ptdf_loss_mode="flows",
    ),

        # ── 7. PTDF flows - increased weight ──
    #PhysicsConfig(
    #    w_phys=0.04,
    #    physics_mode="rich",
    #    use_power_balance=True,
    #    use_angle_ref_penalty=True,
    #    use_ptdf_loss=True,
    #    weight_ptdf=0.3,
    #    ptdf_loss_mode="flows",
    # ),
]

In [ ]:
head_compare_configs = [
    # ── 1. Pure MSE reference (no physics, no PTDF) ──
    PhysicsConfig(
        w_phys=0.0,
        use_ptdf_loss=False,
       weight_ptdf=0.0,
    ),

    # ── 2. Best physics reference (no PTDF) ──
    PhysicsConfig(
        w_phys=0.04,
        physics_mode="rich",
        use_power_balance=True,
        use_angle_ref_penalty=True,
        use_ptdf_loss=False,
        weight_ptdf=0.0,
    ),
    # ── 3. Best physics reference (no PTDF) ──
    #PhysicsConfig(
    #   w_phys=0.06,
    #    physics_mode="rich",
    #    use_power_balance=True,
    #    use_angle_ref_penalty=True,
    #    use_ptdf_loss=False,
    #    weight_ptdf=0.0,
    #),


    # ── 4. PTDF flows mode ──
    PhysicsConfig(
        w_phys=0.04,
        physics_mode="rich",
        use_power_balance=True,
        use_angle_ref_penalty=True,
        use_ptdf_loss=True,
        weight_ptdf=0.1,
        ptdf_loss_mode="flows",
    ),

        # ── 7. PTDF flows - increased weight ──
    #PhysicsConfig(
    #    w_phys=0.04,
    #    physics_mode="rich",
    #    use_power_balance=True,
    #    use_angle_ref_penalty=True,
    #    use_ptdf_loss=True,
    #    weight_ptdf=0.3,
    #    ptdf_loss_mode="flows",
    # ),
]

## IEEE9 Sweep

In [52]:

runs_ieee9_test_phys_vs_ptdf = run_hparam_sweep(
    networks=networks_ieee9_500_singel_slack,
    physics_configs=mixed_configs,
    batch_sizes=[64],
    learning_rates=[5e-4],
    num_epochs=50,
    seeds=[42],
    tag="Standard vs. encoder head",
    verbose=False,
    save_path=os.path.join(TRAINING_RESULTS_DIR, "ieee9_500_single_slack_head_mode_060526.json"),
    head_modes=("standard","with_encoder"),
)

1/2 | mode=rich | cfg=mode-rich_w-0.04_PB_AR_PTDF-flows-pt0.1-lines | wphys=0.04 | ptdfw=0.1 | ptdfmode=flows | ptdfbr=lines | bs=64 | lr=5e-04 | warmup=10 | seed=42 | Starting time=2026-05-06 14:16:00 | remaining=calculating... | finish≈calculating...


INFO:__main__:Precomputing admittance matrices...
INFO:__main__:[YCHECK] Summary (manual): 0 PyPSA Y, 350 manual Y, 0 mismatches.
INFO:__main__:[YCHECK] Summary (manual): 0 PyPSA Y, 75 manual Y, 0 mismatches.
INFO:__main__:[YCHECK] Summary (manual): 0 PyPSA Y, 75 manual Y, 0 mismatches.
INFO:__main__:Y-matrices cached successfully
Training: 100%|██████████| 50/50 [51:00<00:00, 61.21s/epoch]


→ done in 51.6 min | Train 0.0032 Val 0.0046 Test 0.0054 
2/2 | mode=rich | cfg=mode-rich_w-0.04_PB_AR_PTDF-flows-pt0.1-lines | wphys=0.04 | ptdfw=0.1 | ptdfmode=flows | ptdfbr=lines | bs=64 | lr=5e-04 | warmup=10 | seed=42 | Starting time=2026-05-06 15:07:39 | remaining=51m 38s | est_total=1h 43m 16s | finish≈2026-05-06 15:59:17


INFO:__main__:Precomputing admittance matrices...
INFO:__main__:[YCHECK] Summary (manual): 0 PyPSA Y, 350 manual Y, 0 mismatches.
INFO:__main__:[YCHECK] Summary (manual): 0 PyPSA Y, 75 manual Y, 0 mismatches.
INFO:__main__:[YCHECK] Summary (manual): 0 PyPSA Y, 75 manual Y, 0 mismatches.
INFO:__main__:Y-matrices cached successfully
Training: 100%|██████████| 50/50 [50:44<00:00, 60.90s/epoch]


→ done in 51.4 min | Train 0.0043 Val 0.0061 Test 0.0059 


In [ ]:

runs_ieee9_1500_phys_vs_ptdf = run_hparam_sweep(
    networks=networks_ieee9_mega_singel_slack,
    physics_configs=ptdf_comparison_configs,
    batch_sizes=[64],
    learning_rates=[5e-4],
    num_epochs=50,
    seeds=[42],
    tag="MSE, Physics and PTDF Sweep",
    verbose=False,
    save_path=os.path.join(TRAINING_RESULTS_DIR, "ieee9_mega_single_slack_ptdf_sweep_0204526.json"),
)

In [21]:
logger.setLevel(logging.ERROR)
runs_ieee9_1500_phys_vs_ptdf = run_hparam_sweep(
    networks=networks_ieee9_mega_singel_slack,
    physics_configs=ptdf_comparison_configs,
    batch_sizes=[64],
    learning_rates=[5e-4],
    num_epochs=50,
    seeds=[42],
    tag="MSE, Physics and PTDF Sweep",
    verbose=False,
    save_path=os.path.join(TRAINING_RESULTS_DIR, "ieee9_mega_single_slack_ptdf_sweep_0204526.json"),
)

1/5 | mode=rich | cfg=mode-rich_w-0.0_PB | wphys=0.0 | ptdfw=0.0 | ptdfmode=matrix | ptdfbr=lines | bs=64 | lr=5e-04 | warmup=10 | seed=42 | Starting time=2026-05-02 23:35:09 | remaining=calculating... | finish≈calculating...


Training: 100%|██████████| 50/50 [1:41:57<00:00, 122.35s/epoch]


→ done in 104.0 min | Train 0.0001 Val 0.0004 Test 0.0001 
2/5 | mode=rich | cfg=mode-rich_w-0.04_PB_AR | wphys=0.04 | ptdfw=0.0 | ptdfmode=matrix | ptdfbr=lines | bs=64 | lr=5e-04 | warmup=10 | seed=42 | Starting time=2026-05-03 01:19:09 | remaining=6h 55m 58s | est_total=8h 39m 58s | finish≈2026-05-03 08:15:08


Training: 100%|██████████| 50/50 [2:27:14<00:00, 176.69s/epoch]  


→ done in 149.3 min | Train 0.0003 Val 0.0003 Test 0.0004 
3/5 | mode=rich | cfg=mode-rich_w-0.06_PB_AR | wphys=0.06 | ptdfw=0.0 | ptdfmode=matrix | ptdfbr=lines | bs=64 | lr=5e-04 | warmup=10 | seed=42 | Starting time=2026-05-03 03:48:29 | remaining=7h 27m 59s | est_total=12h 26m 39s | finish≈2026-05-03 11:16:29


...

→ done in 138.8 min | Train 0.0004 Val 0.0005 Test 0.0004 
4/5 | mode=rich | cfg=mode-rich_w-0.04_PB_AR_PTDF-flows-pt0.1-lines | wphys=0.04 | ptdfw=0.1 | ptdfmode=flows | ptdfbr=lines | bs=64 | lr=5e-04 | warmup=10 | seed=42 | Starting time=2026-05-03 06:07:18 | remaining=4h 37m 37s | est_total=11h 34m 3s | finish≈2026-05-03 10:44:55


Training: 100%|██████████| 50/50 [2:28:28<00:00, 178.16s/epoch]  


→ done in 150.5 min | Train 0.0004 Val 0.0005 Test 0.0005 
5/5 | mode=rich | cfg=mode-rich_w-0.04_PB_AR_PTDF-flows-pt0.3-lines | wphys=0.04 | ptdfw=0.3 | ptdfmode=flows | ptdfbr=lines | bs=64 | lr=5e-04 | warmup=10 | seed=42 | Starting time=2026-05-03 08:37:45 | remaining=2h 29m 19s | est_total=12h 26m 39s | finish≈2026-05-03 11:07:05


Training: 100%|██████████| 50/50 [2:27:18<00:00, 176.77s/epoch]  


→ done in 149.2 min | Train 0.0005 Val 0.0006 Test 0.0006 


In [25]:
save_sweep_results(
    runs_ieee9_1500_phys_vs_ptdf,
    save_dir=TRAINING_RESULTS_DIR,
    tag="ieee9_mega_single_slack_ptdf_sweep",
    save_models=True,   
    save_csv=True,
    save_histories=True,  
)

  [OK] Full run list  -> C:\Users\STSI\OneDrive - USN\Data_PF_GNN\training_results_saved\runs_ieee9_mega_single_slack_ptdf_sweep_20260503_112216.pkl
  [OK] Metrics CSV    -> C:\Users\STSI\OneDrive - USN\Data_PF_GNN\training_results_saved\runs_ieee9_mega_single_slack_ptdf_sweep_20260503_112216_metrics.csv
  [OK] Histories      -> C:\Users\STSI\OneDrive - USN\Data_PF_GNN\training_results_saved\runs_ieee9_mega_single_slack_ptdf_sweep_20260503_112216_histories/  (5 files)
  [OK] Model weights  -> C:\Users\STSI\OneDrive - USN\Data_PF_GNN\training_results_saved\models\runs_ieee9_mega_single_slack_ptdf_sweep_20260503_112216/  (5 files)
  [OK] Manifest       -> C:\Users\STSI\OneDrive - USN\Data_PF_GNN\training_results_saved\runs_ieee9_mega_single_slack_ptdf_sweep_20260503_112216_manifest.json

  Saved 5 runs to 'C:\Users\STSI\OneDrive - USN\Data_PF_GNN\training_results_saved/runs_ieee9_mega_single_slack_ptdf_sweep_20260503_112216*'


In [ ]:
runs_ieee9_500_phys_vs_ptdf=load_sweep_results(os.path.join(TRAINING_RESULTS_DIR, "ieee9_large_single_slack_ptdf_sweep_060426.json"))

## CIGRE14 Sweep

In [ ]:
runs_cigre14_500_phys_vs_ptdf = run_hparam_sweep(
    networks=networks_cigre14_large_singel_slack,
    physics_configs=ptdf_comparison_configs,
    batch_sizes=[64],
    learning_rates=[5e-4],
    num_epochs=50,
    seeds=[42],
    tag="MSE, Physics and PTDF Sweep",
    verbose=False,
    save_path="training_results_saved\\cigre14_large_single_slack_ptdf_sweep_060426.json",
)

In [ ]:
runs_cigre14_500_phys_vs_ptdf =load_sweep_results("training_results_saved\\cigre14_large_single_slack_ptdf_sweep_060426.json")

In [ ]:
runs_cigre14_750_phys_vs_ptdf = run_hparam_sweep(
    networks=networks_cigre14_xlarge_singel_slack,
    physics_configs=CIGRE_14_large_configs,
    batch_sizes=[64],
    learning_rates=[5e-4],
    num_layers_list=[3,5],
    num_epochs=50,
    seeds=[42],
    tag="CIGRE_larger_dataset",
    verbose=False,
    save_path="training_results_saved\\cigre14_xlarge_single_slack_ptdf_sweep_110426.json",
)

## IEEE30 Sweep

In [ ]:
runs_ieee30_1000_phys_vs_ptdf = run_hparam_sweep(
    networks=networks_ieee30_xlarge_singel_slack,
    physics_configs=ptdf_comparison_configs2 ,
    batch_sizes=[64],
    learning_rates=[5e-4],
    num_epochs=50,
    num_layers_list=[5],
    seeds=[42],
    tag="MSE, Physics and PTDF Sweep",
    verbose=False,
    save_path="training_results_saved\\ieee30_xlarge_single_slack_ptdf_sweep_210426.json",
)

In [ ]:
runs_ieee30_500_phys_vs_ptdf=load_sweep_results("training_results_saved\\ieee30_large_single_slack_ptdf_sweep_140426.json")

In [ ]:
runs_ieee30_500_head = run_hparam_sweep(
    networks=networks_ieee30_500_wide,
    physics_configs=head_compare_configs,
    batch_sizes=[64],
    learning_rates=[5e-4],
    num_epochs=50,
    num_layers_list=[5],
    seeds=[42],
    tag="standard vs. encoder head",
    verbose=False,
    save_path=os.path.join(TRAINING_RESULTS_DIR, "ieee30_500_wide_head_compare_210426.json"),
)

## Mixed dataset sweep

In [16]:
# ── Sample 500 networks from each system (reproducible) ──────────────────────
_rng = random.Random(42)

def _sample(networks, n, seed=42):
    rng = random.Random(seed)
    pool = list(networks)
    rng.shuffle(pool)
    return pool[:n]

networks_mixed_1500 = (
    _sample(networks_ieee9_mega_singel_slack,   500) +
    _sample(networks_cigre14_mega_singel_slack, 500) +
    _sample(networks_ieee30_mega_singel_slack,  500)
)
random.Random(42).shuffle(networks_mixed_1500)  # interleave systems

print(f"Mixed dataset: {len(networks_mixed_1500)} networks")
print(f"  bus counts: {sorted(set(len(n.buses) for n in networks_mixed_1500))}")

Mixed dataset: 1500 networks
  bus counts: [9, 10, 15, 16, 17, 18, 30, 31, 32, 33, 34, 35]


In [37]:
# save the mixed dataset for later use
save_dataset_list(networks_mixed_1500, os.path.join(TRAINING_NETWORKS_DIR, "mixed_1500.json"),subfolder="mixed_1500_single_slack")

Saved 1500 networks → C:\Users\STSI\OneDrive - USN\Data_PF_GNN\training_networks_saved\mixed_1500_single_slack/ (index: C:\Users\STSI\OneDrive - USN\Data_PF_GNN\training_networks_saved\mixed_1500.json)


'C:\\Users\\STSI\\OneDrive - USN\\Data_PF_GNN\\training_networks_saved\\mixed_1500.json'

In [16]:
networks_mixed_1500=load_dataset_list(os.path.join(TRAINING_NETWORKS_DIR, "mixed_1500.json"))

Loaded 1500 networks from C:\Users\STSI\OneDrive - USN\Data_PF_GNN\training_networks_saved\mixed_1500.json


In [19]:
runs_mixed_1500_phys_vs_ptdf = run_hparam_sweep(
    networks=networks_mixed_1500,
    physics_configs=mixed_configs,
    batch_sizes=[16,32],
    learning_rates=[5e-4],
    num_epochs=50,
    num_layers_list=[3],
    seeds=[42],
    tag="Mixed_1500_Physics_vs_PTDF",
    verbose=False,
    save_path=os.path.join(TRAINING_RESULTS_DIR, "mixed_1500_phys_vs_ptdf_sweep.json"),
)

[checkpoint] Loaded 5 completed runs; 5 keys will be skipped.
[checkpoint] Restored 5 prior training_time values (avg 246.4 min/run).
1/2 | mode=rich | cfg=mode-rich_w-0.04_PB_AR_PTDF-flows-pt0.1-lines | wphys=0.04 | ptdfw=0.1 | ptdfmode=flows | ptdfbr=lines | bs=16 | lr=5e-04 | warmup=10 | seed=42 | Starting time=2026-05-04 15:05:28 | remaining=8h 10m 27s | est_total=8h 10m 27s | finish≈2026-05-04 23:15:56


INFO:__main__:Precomputing admittance matrices...
INFO:__main__:[YCHECK] Summary (manual): 0 PyPSA Y, 1050 manual Y, 0 mismatches.
INFO:__main__:[YCHECK] Summary (manual): 0 PyPSA Y, 225 manual Y, 0 mismatches.
INFO:__main__:[YCHECK] Summary (manual): 0 PyPSA Y, 225 manual Y, 0 mismatches.
INFO:__main__:Y-matrices cached successfully
Training: 100%|██████████| 50/50 [7:25:07<00:00, 534.15s/epoch]   


→ done in 448.7 min | Train 77.3584 Val 0.8653 Test 0.1477 
2/2 | mode=rich | cfg=mode-rich_w-0.04_PB_AR_PTDF-flows-pt0.1-lines | wphys=0.04 | ptdfw=0.1 | ptdfmode=flows | ptdfbr=lines | bs=32 | lr=5e-04 | warmup=10 | seed=42 | Starting time=2026-05-04 22:34:08 | remaining=4h 19m 39s | est_total=8h 39m 19s | finish≈2026-05-05 02:53:48


INFO:__main__:Precomputing admittance matrices...
INFO:__main__:[YCHECK] Summary (manual): 0 PyPSA Y, 1050 manual Y, 0 mismatches.
INFO:__main__:[YCHECK] Summary (manual): 0 PyPSA Y, 225 manual Y, 0 mismatches.
INFO:__main__:[YCHECK] Summary (manual): 0 PyPSA Y, 225 manual Y, 0 mismatches.
INFO:__main__:Y-matrices cached successfully
Training: 100%|██████████| 50/50 [4:56:58<00:00, 356.37s/epoch]  


→ done in 300.3 min | Train 78.9856 Val 31.4001 Test 0.1745 


In [20]:
save_sweep_results(
    runs_mixed_1500_phys_vs_ptdf,
    save_dir=TRAINING_RESULTS_DIR,
    tag="mixed_mega_single_slack_ptdf_sweep",
    save_models=True,   
    save_csv=True,
    save_histories=True,  
)

  [OK] Full run list  -> C:\Users\STSI\OneDrive - USN\Data_PF_GNN\training_results_saved\runs_mixed_mega_single_slack_ptdf_sweep_20260505_064201.pkl
  [OK] Metrics CSV    -> C:\Users\STSI\OneDrive - USN\Data_PF_GNN\training_results_saved\runs_mixed_mega_single_slack_ptdf_sweep_20260505_064201_metrics.csv
  [OK] Histories      -> C:\Users\STSI\OneDrive - USN\Data_PF_GNN\training_results_saved\runs_mixed_mega_single_slack_ptdf_sweep_20260505_064201_histories/  (7 files)
  [OK] Model weights  -> C:\Users\STSI\OneDrive - USN\Data_PF_GNN\training_results_saved\models\runs_mixed_mega_single_slack_ptdf_sweep_20260505_064201/  (7 files)
  [OK] Manifest       -> C:\Users\STSI\OneDrive - USN\Data_PF_GNN\training_results_saved\runs_mixed_mega_single_slack_ptdf_sweep_20260505_064201_manifest.json

  Saved 7 runs to 'C:\Users\STSI\OneDrive - USN\Data_PF_GNN\training_results_saved/runs_mixed_mega_single_slack_ptdf_sweep_20260505_064201*'


#### testing dangeling bus impact